In [ ]:
import sys
sys.path.append(".")  
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 18,
    "font.size": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

#### Example 01 - Gravitational Field of a Simple Tetrahedron

In [ ]:
# ===============================================================
# Example 01 — Gravitational Field of a Simple Tetrahedron
# ===============================================================
# This example demonstrates how to use the PolyhedronGravitation
# class from the 'polygrav' package to compute:
#   - Gravitational potential (U)
#   - Gravitational acceleration (g)
#   - Gravity gradient tensor (Γ)
# for selected evaluation points.

# ---------------------------------------------------------------
# 1. Import required modules
# ---------------------------------------------------------------
import numpy as np
from polygravitation import PolyhedronGravitation

# ---------------------------------------------------------------
# 2. Define the polyhedron geometry (a simple tetrahedron)
# ---------------------------------------------------------------
vertices = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=float)

faces = np.array([
    [0, 2, 1],
    [0, 1, 3],
    [2, 3, 0],
    [1, 2, 3]
], dtype=np.int32)

# ---------------------------------------------------------------
# 3. Instantiate the PolyhedronGravitation model
# ---------------------------------------------------------------
model = PolyhedronGravitation(vertices=vertices,
                              faces=faces,
                              G=1.0,
                              density=1.0,
                              eps=0.0,
                              orient_faces=True)

# ---------------------------------------------------------------
# 4. Define test points
# ---------------------------------------------------------------
test_points = [
    np.array([2.0, 2.0, 2.0]),    # Exterior point
    np.array([0.25, 0.25, 0.25]), # Interior point
    np.array([0.0, 0.0, 0.0]),     # Vertex point
    np.array([0.0, 1.0, 0.0]),
    np.array([0.0, 1.0000000001, 0.0])
]

# ---------------------------------------------------------------
# 5. Compute and save results
# ---------------------------------------------------------------
output_filename = "gravitation_results.txt"

with open(output_filename, "w") as f:
    f.write("--- Gravitational Field Calculations for a Tetrahedron ---\n")

    for p in test_points:
        # Perform calculations
        potential = model.potential(p)
        acceleration = model.acceleration(p)
        tensor = model.gravity_tensor(p)

        # Write organized output
        f.write("\n" + "="*60 + "\n")
        f.write(f"Results for Point: [{p[0]:.2f}, {p[1]:.2f}, {p[2]:.2f}]\n")
        f.write("="*60 + "\n")

        # Potential
        f.write(f"\nPotential U: {potential:.12f}\n")

        # Acceleration
        f.write("\nAcceleration g:\n")
        f.write(f"  g_x: {acceleration[0]: .12e}\n")
        f.write(f"  g_y: {acceleration[1]: .12e}\n")
        f.write(f"  g_z: {acceleration[2]: .12e}\n")

        # Gravity Gradient Tensor
        f.write("\nGravity Gradient Tensor Γ:\n")
        for i in range(3):
            f.write(f"  [{tensor[i, 0]:>16.12e} {tensor[i, 1]:>16.12e} {tensor[i, 2]:>16.12e}]\n")

print(f"Calculations complete. Results saved to '{output_filename}'")

# ---------------------------------------------------------------
# 6. Close the model (shut down thread pool)
# ---------------------------------------------------------------
model.close()


#### Example 02 — Laplacian Slice of a Tetrahedron

In [ ]:
# ===============================================================
# Example 02 — Laplacian Slice of a Tetrahedron
# ===============================================================
# This example computes and visualizes the Laplacian (∇²U / (Gρ))
# on a 2D plane slice through a tetrahedral mass using the
# analytical polyhedron gravity model.
#
# Demonstrates:
#   - Using gravity_tensor() to derive the Laplacian analytically
#   - Generating a color map of ∇²U inside/outside the body
#   - Drawing the tetrahedron cross-section overlay
#
# Expected Results:
#   Inside the tetrahedron: ∇²U / (Gρ) ≈ -4π
#   Outside the body:      ∇²U / (Gρ) ≈ 0
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# ---------------------------------------------------------------
# 1. Import the polyhedron gravity model
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation

# ---------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------
def _cross(o, a, b):
    return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

def convex_hull_2d(points_xy):
    """Compute convex hull (monotone chain) for plotting cross-section."""
    P = np.asarray(points_xy, float)
    if P.shape[0] < 3:
        return P.copy()
    pts = P[np.lexsort((P[:, 1], P[:, 0]))]
    lower = []
    for p in pts:
        while len(lower) >= 2 and _cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(tuple(p))
    upper = []
    for p in pts[::-1]:
        while len(upper) >= 2 and _cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(tuple(p))
    return np.array(lower[:-1] + upper[:-1], dtype=float)

# ---------------------------------------------------------------
# Geometry and Model
# ---------------------------------------------------------------
vertices = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=float)

faces = np.array([
    [0, 2, 1],
    [0, 1, 3],
    [0, 3, 2],
    [1, 2, 3]
], dtype=np.int32)

model = PolyhedronGravitation(vertices=vertices, faces=faces, G=1.0, density=1.0)
print("PolyhedronGravitation model initialized.")

# ---------------------------------------------------------------
# Compute Laplacian Field
# ---------------------------------------------------------------
z_fixed = 0.25
x_range, y_range = (-0.2, 1.0), (-0.2, 1.0)
resolution = 200

xs = np.linspace(*x_range, resolution)
ys = np.linspace(*y_range, resolution)
X, Y = np.meshgrid(xs, ys, indexing='xy')
grid_points = np.stack([X.ravel(), Y.ravel(), np.full(X.size, z_fixed)], axis=1)

print(f"Computing Laplacian on a {resolution}×{resolution} grid at z={z_fixed}...")

tensors_ana = model.gravity_tensor(grid_points)
laplacian_ana = np.trace(tensors_ana, axis1=1, axis2=2)
L_ana = laplacian_ana.reshape(X.shape) / (model.G * model.rho)
print("Laplacian computation complete.")

# ---------------------------------------------------------------
# Compute Boundary Intersection
# ---------------------------------------------------------------
def edge_intersection(v1, v2, z_plane):
    z1, z2 = v1[2], v2[2]
    if (z1 > z_plane and z2 < z_plane) or (z1 < z_plane and z2 > z_plane):
        t = (z_plane - z1) / (z2 - z1)
        return v1 + t * (v2 - v1)
    return None

edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
intersection_points = [
    pt[:2] for i, j in edges
    if (pt := edge_intersection(vertices[i], vertices[j], z_fixed)) is not None
]

boundary_xy = convex_hull_2d(intersection_points)
boundary_xy = np.vstack([boundary_xy, boundary_xy[0]])

# ---------------------------------------------------------------
# Plot (LaTeX enabled)
# ---------------------------------------------------------------
N_colors = 50
colors = [(0.1, 0.4, 1)] + [
    (i/(N_colors-3), 1, 1 - i/(N_colors-3))
    for i in range(N_colors-2)
] + [(1, 0, 0)]
cmap_discrete = ListedColormap(colors)
vmin, vmax = -4*np.pi, 0.0
bounds = np.linspace(vmin, vmax, N_colors + 1)
norm = BoundaryNorm(bounds, cmap_discrete.N)

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
fig.suptitle(r"Analytical Laplacian $\nabla^2 U / (G\rho)$ at $z = 0.25$", fontsize=16)

im = ax.imshow(
    L_ana,
    extent=(x_range[0], x_range[1], y_range[0], y_range[1]),
    origin='lower',
    cmap=cmap_discrete,
    norm=norm
)

ax.set_title(r"Calculated from trace of analytical tensor", fontsize=12)
ax.set_xlabel(r"$x$-axis")
ax.set_ylabel(r"$y$-axis")

# Overlay boundary
ax.plot(boundary_xy[:, 0], boundary_xy[:, 1],
        color='white', linewidth=2, linestyle='-')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.savefig("laplacian_tetra_slice_analytical.png", dpi=200)
plt.show()
print("Plot saved to 'laplacian_tetra_slice_analytical.png'.")

# ---------------------------------------------------------------
# Cleanup
# ---------------------------------------------------------------
model.close()



#### Example 03 — Analytical Laplacian of a Concave L-Shape

In [ ]:
# ===============================================================
# Example 03 — Analytical Laplacian of a Concave L-Shape
# ===============================================================
# This example builds a concave polyhedron (“L”-shaped block made
# of three unit cubes) and computes the analytical Laplacian field
# ∇²U / (Gρ) on a horizontal plane slice.
#
# Demonstrates:
#   • Procedural mesh generation for a concave geometry
#   • Analytical Laplacian derived from gravity tensor trace
#   • Visualization of interior/exterior Laplacian regions
#
# Expected results:
#   Inside the solid: ∇²U / (Gρ) ≈ −4π
#   Outside the body: ≈ 0
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from polygravitation import PolyhedronGravitation   # Correct module

# ---------------------------------------------------------------
# 1. Build a Concave L-Shaped Mesh
# ---------------------------------------------------------------
def build_L_mesh():
    """
    Construct a concave L-shaped polyhedron from three adjacent cubes.
    Emits only outward-oriented external triangular faces.
    """
    # Occupied cube coordinates
    occ = {(0,0,0), (1,0,0), (0,1,0)}

    # Cube face directions
    dirs = [
        ("-x", (-1,0,0), np.array([-1,0,0])), ("+x", (1,0,0), np.array([1,0,0])),
        ("-y", (0,-1,0), np.array([0,-1,0])), ("+y", (0,1,0), np.array([0,1,0])),
        ("-z", (0,0,-1), np.array([0,0,-1])), ("+z", (0,0,1), np.array([0,0,1])),
    ]

    def face_quad(x, y, z, name):
        if name == "-x": return [(x,y,z), (x,y,z+1), (x,y+1,z+1), (x,y+1,z)]
        if name == "+x": return [(x+1,y,z), (x+1,y+1,z), (x+1,y+1,z+1), (x+1,y,z+1)]
        if name == "-y": return [(x,y,z), (x+1,y,z), (x+1,y,z+1), (x,y,z+1)]
        if name == "+y": return [(x,y+1,z), (x,y+1,z+1), (x+1,y+1,z+1), (x+1,y+1,z)]
        if name == "-z": return [(x,y,z), (x,y+1,z), (x+1,y+1,z), (x+1,y,z)]
        if name == "+z": return [(x,y,z+1), (x+1,y,z+1), (x+1,y+1,z+1), (x,y+1,z+1)]

    verts, vidx = [], {}
    def get_idx(p):
        if p not in vidx:
            vidx[p] = len(verts)
            verts.append(np.array(p, dtype=float))
        return vidx[p]

    faces = []
    for (x, y, z) in occ:
        for name, delta, outward in dirs:
            nx, ny, nz = x+delta[0], y+delta[1], z+delta[2]
            if (nx, ny, nz) in occ:
                continue  # skip internal face
            quad = face_quad(x, y, z, name)
            triA = [get_idx(quad[0]), get_idx(quad[1]), get_idx(quad[2])]
            triB = [get_idx(quad[0]), get_idx(quad[2]), get_idx(quad[3])]
            for tri in (triA, triB):
                p0, p1, p2 = (verts[tri[0]], verts[tri[1]], verts[tri[2]])
                n = np.cross(p1 - p0, p2 - p0)
                if np.dot(n, outward) < 0:
                    tri[1], tri[2] = tri[2], tri[1]
                faces.append(tuple(tri))

    return np.array(verts, float), np.array(faces, np.int32)

V, F = build_L_mesh()
print(f"Concave L-mesh created: {len(V)} vertices, {len(F)} faces")

# ---------------------------------------------------------------
# 2. Instantiate Gravity Model
# ---------------------------------------------------------------
model = PolyhedronGravitation(vertices=V, faces=F, G=1.0, density=1.0)
print("PolyhedronGravitation model initialized.")

# ---------------------------------------------------------------
# 3. Analytical Laplacian on a 2D Slice
# ---------------------------------------------------------------
z_fixed = 0.5
x_range, y_range = (-0.5, 2.5), (-0.5, 2.5)
resolution = 300

xs = np.linspace(*x_range, resolution)
ys = np.linspace(*y_range, resolution)
X, Y = np.meshgrid(xs, ys, indexing='xy')
grid_points = np.stack([X.ravel(), Y.ravel(), np.full(X.size, z_fixed)], axis=1)
print(f"Computing Laplacian on a {resolution}×{resolution} grid at z={z_fixed}...")

tensors = model.gravity_tensor(grid_points)
laplacian = np.trace(tensors, axis1=1, axis2=2)
L = laplacian.reshape(X.shape) / (model.G * model.rho)
print("Laplacian computation complete.")

# ---------------------------------------------------------------
# 4. Extract Cross-Section Boundary
# ---------------------------------------------------------------
def section_segments_from_mesh(Vm, Fm, z0, tol=0):
    segs = []
    for i, j, k in Fm:
        tri = np.vstack([Vm[i], Vm[j], Vm[k]])
        z = tri[:, 2]
        pts = []
        for a, b in ((0,1), (1,2), (2,0)):
            z1, z2 = z[a], z[b]
            if (z1 - z0) * (z2 - z0) < -tol:
                t = (z0 - z1) / (z2 - z1)
                p = tri[a] + t * (tri[b] - tri[a])
                pts.append(p[:2])
        if len(pts) == 2:
            segs.append((tuple(pts[0]), tuple(pts[1])))
    return segs

def stitch_segments_to_polylines(segs, tol=1e-9):
    if not segs:
        return []
    def key(p): return (round(p[0]/tol), round(p[1]/tol))
    pts, adj = {}, {}
    for a, b in segs:
        ka, kb = key(a), key(b)
        if ka not in pts: pts[ka] = np.array(a)
        if kb not in pts: pts[kb] = np.array(b)
        adj.setdefault(ka, []).append(kb)
        adj.setdefault(kb, []).append(ka)
    visited, polys = set(), []
    for s in list(adj.keys()):
        if s in visited: continue
        line = [pts[s]]
        prev, cur = s, adj[s][0]
        while cur != s:
            visited.add(cur)
            line.append(pts[cur])
            nxt = [k for k in adj[cur] if k != prev]
            if not nxt: break
            prev, cur = cur, nxt[0]
        line.append(pts[s])
        polys.append(np.vstack(line))
    return polys

segs = section_segments_from_mesh(V, F, z_fixed)
polys = stitch_segments_to_polylines(segs)

# ---------------------------------------------------------------
# 5. Plot Analytical Laplacian Field
# ---------------------------------------------------------------
N_colors = 50
colors = [(0.1, 0.4, 1)] + [
    (i/(N_colors-3), 1, 1 - i/(N_colors-3)) for i in range(N_colors-2)
] + [(1, 0, 0)]
cmap_discrete = ListedColormap(colors)
vmin, vmax = -4*np.pi, 0.0
bounds = np.linspace(vmin, vmax, N_colors + 1)
norm = BoundaryNorm(bounds, cmap_discrete.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(L, extent=(*x_range, *y_range), origin='lower',
               cmap=cmap_discrete, norm=norm)

# --- Colorbar ---
cbar = plt.colorbar(im, ax=ax, boundaries=bounds)
cbar.set_label(r"$\nabla^2 U / (G\,\rho)$", fontsize=20)
cbar.set_ticks(np.arange(int(vmin)//2*2, int(vmax)+1, 2))
cbar.ax.set_yticklabels([str(i) for i in np.arange(int(vmin)//2*2, int(vmax)+1, 2)])

# Overlay boundary
for poly in polys:
    ax.plot(poly[:,0], poly[:,1], color='white', linewidth=2, linestyle='-')

ax.set_title(r"Analytical $\nabla^2 U$ of a Concave Body at $z = 0.5$", fontsize=23)
ax.set_xlabel(r"$x$-axis")
ax.set_ylabel(r"$y$-axis")

plt.tight_layout()
plt.savefig("concave_L_mesh_laplacian_analytical.png", dpi=200)
plt.show()
print("Plot saved to 'concave_L_mesh_laplacian_analytical.png'.")

# ---------------------------------------------------------------
# 6. Sanity Check — Potential at Sample Points
# ---------------------------------------------------------------
pts_to_check = np.array([
    [0.5, 0.5, 0.5],   # inside
    [1.5, 1.5, 0.5],   # cavity
    [3.0, 3.0, 2.0],   # far field
], float)

U = model.potential(pts_to_check)
print("\n--- Potential at Specific Points ---")
for p, u in zip(pts_to_check, U):
    print(f"  U({p}) = {u:.8f}")

# ---------------------------------------------------------------
# 7. Cleanup
# ---------------------------------------------------------------
model.close()


#### Example 04 - Analytical Laplacian of a Torus

In [ ]:
# ===============================================================
# Example 04 — Analytical Laplacian of a Torus
# ===============================================================
# This example constructs a closed triangular mesh for a torus
# and computes the analytical Laplacian field ∇²U / (Gρ)
# on the mid-plane (z = 0) using the polyhedron gravity model.
#
# Demonstrates:
#   • Parametric torus mesh generation
#   • Handling of non-star-shaped geometry (orient_faces = False)
#   • Analytical Laplacian visualization with LaTeX formatting
#
# Expected results:
#   Inside solid ring: ∇²U / (Gρ) ≈ −4π
#   Outside material:  ∇²U / (Gρ) ≈ 0
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import ListedColormap, BoundaryNorm

from polygravitation import PolyhedronGravitation   

# ---------------------------------------------------------------
# 1. Torus Mesh Generation
# ---------------------------------------------------------------
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Generate a watertight triangular mesh for a torus."""
    u = np.linspace(0, 2*np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2*np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, np.int32)

# ---------------------------------------------------------------
# 2. Geometry, Grid, and Model Setup
# ---------------------------------------------------------------
R1, R2 = 2.0, 1.0
a = R1 + R2 + 1.0

vertices, faces = create_torus_tri_mesh(R1, R2, n_u=100, n_v=50)

# Non-star-shaped geometry → disable face orientation
model = PolyhedronGravitation(vertices=vertices, faces=faces,
                              density=1.0, G=1.0, orient_faces=False)
print("PolyhedronGravitation torus model initialized.")

grid_res = 251
x_range = np.linspace(-a, a, grid_res)
y_range = np.linspace(-a, a, grid_res)
XX, YY = np.meshgrid(x_range, y_range)
grid_points = np.stack([XX.ravel(), YY.ravel(),
                        np.zeros(grid_res**2)], axis=1)

# ---------------------------------------------------------------
# 3. Compute Analytical Laplacian
# ---------------------------------------------------------------
print(f"Computing Laplacian on {grid_res}×{grid_res} grid (z = 0)...")

tensors = model.gravity_tensor(grid_points)

# ∇²U = −trace(Γ)
laplacian_values = -np.trace(tensors, axis1=1, axis2=2)
laplacian_grid = laplacian_values.reshape(grid_res, grid_res)

print("Laplacian computation complete.")

# ---------------------------------------------------------------
# 4. Plotting with Custom Style and LaTeX Labels
# ---------------------------------------------------------------
N = 50
colors = [(0.1, 0.4, 1)] + [(i/(N-3), 1.0, 1.0 - i/(N-3)) for i in range(N-2)] + [(1, 0, 0)]
cmap_discrete = ListedColormap(colors)

vmin, vmax = -4*np.pi, 0.0
bounds = np.linspace(vmin, vmax, N+1)
norm = BoundaryNorm(bounds, cmap_discrete.N)

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(laplacian_grid, extent=[-a, a, -a, a],
               origin='lower', cmap=cmap_discrete,
               norm=norm, interpolation='nearest')

# --- Colorbar ---
cbar = plt.colorbar(im, ax=ax, boundaries=bounds,
                    fraction=0.046, pad=0.04)
cbar.set_label(r"$\nabla^2 U / (G\rho)$", fontsize=18)
cbar.set_ticks([0, -2, -4, -6, -8, -10, -12])
cbar.ax.set_yticklabels([str(i) for i in [0, -2, -4, -6, -8, -10, -12]])

# --- Torus boundaries (white circles) ---
inner_r, outer_r = R1 - R2, R1 + R2
ax.add_patch(Circle((0, 0), inner_r, edgecolor='white',
                    facecolor='none', linewidth=2))
ax.add_patch(Circle((0, 0), outer_r, edgecolor='white',
                    facecolor='none', linewidth=2))

# --- Axis labels and annotations ---
ax.set_title(r"Analytical $\nabla^2 U / (G\rho)$ in plane $z = 0$",
             fontsize=20)
ax.set_xlabel(r"$x$-axis", fontsize=22)
ax.set_ylabel(r"$y$-axis", fontsize=22)
ax.set_aspect('equal')
ax.grid(False)

ax.text(R1, 0, r"$\nabla^2 U = -4\pi$", color='white',
        fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(0, 0, r"$\nabla^2 U = 0$", color='white',
        fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(a - 2, 3, r"$\nabla^2 U = 0$", color='white',
        fontsize=16, ha='center', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig("laplacian_torus_final_integer_ticks.png", dpi=300)
plt.show()
print("Plot saved to 'laplacian_torus_final_integer_ticks.png'.")

# ---------------------------------------------------------------
# 5. Cleanup
# ---------------------------------------------------------------
model.close()


#### Example 05 - Analytical Laplacian of a Torus - Vertical

In [ ]:
# ===============================================================
# Example_05_Torus_Laplacian_VerticalSlice.py
# Analytical Laplacian (∇²U) of a Torus — Vertical (y–z) Plane Slice
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import ListedColormap, BoundaryNorm

# ----------------------------------------------------------------
# Import the machine-precision gravity model
# ----------------------------------------------------------------
from polygravitation import PolyhedronGravitation 

# ==============================================================================
# 1. Torus Mesh Generation
# ==============================================================================
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Creates a perfectly closed (watertight) triangular mesh for a torus."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2 * np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, dtype=np.int32)

# ==============================================================================
# 2. Setup Geometry, Grid, and Model
# ==============================================================================
R1, R2 = 2.0, 1.0
vertices, faces = create_torus_tri_mesh(R1, R2, n_u=100, n_v=50)

# Instantiate the model with face orientation turned OFF for the non-star-shaped torus
model = PolyhedronGravitation(vertices=vertices, faces=faces, G=1.0, density=1.0, orient_faces=False)
print("Model initialized.")

# --- Define the VERTICAL calculation grid (y-z plane at x=0) ---
grid_res = 250
y_limit = R1 + R2 + 0.5
z_limit = R2 + 0.5

y_range = np.linspace(-y_limit, y_limit, grid_res)
z_range = np.linspace(-z_limit, z_limit, grid_res)
YY, ZZ = np.meshgrid(y_range, z_range)

grid_points = np.stack([np.zeros(grid_res**2), YY.ravel(), ZZ.ravel()], axis=1)
print(f"Defined a {256}x{126} calculation grid on the x=0 plane.")

# ==============================================================================
# 3. Compute the Correct Laplacian
# ==============================================================================
print("Computing Laplacian on the vertical grid...")
tensors = model.gravity_tensor(grid_points)
laplacian_values = -np.trace(tensors, axis1=1, axis2=2)
laplacian_grid = laplacian_values.reshape(grid_res, grid_res)
print(" Computation complete.")


# ==============================================================================
# 4. Plotting with Custom Style
# ==============================================================================
# --- Custom colormap and normalization ---
N = 50
colors = [(0.1, 0.4, 1)] + [(i/(N-3), 1, 1 - i/(N-3)) for i in range(N-2)] + [(1, 0, 0)]
cmap_discrete = ListedColormap(colors)
vmin, vmax = -4*np.pi, 0.0
bounds = np.linspace(vmin, vmax, N+1)
norm = BoundaryNorm(bounds, cmap_discrete.N)

# --- Create the plot ---
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(laplacian_grid,
               extent=[-3.5, 3.5, -1.5, 1.5],   # fixed axes
               origin='lower', cmap=cmap_discrete, norm=norm, interpolation='nearest')

# --- Add Colorbar ---
cbar = plt.colorbar(im, ax=ax, boundaries=bounds, fraction=0.046, pad=0.04, shrink=0.40)
cbar.set_ticks([0, -4, -8, -12])
cbar.set_label(r'$\nabla^2 U / (G\rho)$', fontsize=14)

# --- Plot torus boundary (two circles for the vertical slice) ---
circle_left = Circle((-R1, 0), R2, edgecolor='white', facecolor='none', linestyle='-', linewidth=2)
circle_right = Circle((R1, 0), R2, edgecolor='white', facecolor='none', linestyle='-', linewidth=2)
ax.add_patch(circle_left)
ax.add_patch(circle_right)

# --- Labels and Annotations ---
ax.set_title(r"Analytical Laplacian on a 251 × 126 grid in the plane $x = 0$", fontsize=20)
ax.set_xlabel('y-axis', fontsize=20)
ax.set_ylabel('z-axis', fontsize=20)
ax.set_aspect('equal')
ax.grid(False)

ax.text(-R1, 0, r'$\nabla^2 U = -4\pi$', color='white', fontsize=16,
        ha='center', va='center', fontweight='bold')
ax.text( R1, 0, r'$\nabla^2 U = -4\pi$', color='white', fontsize=16,
        ha='center', va='center', fontweight='bold')
ax.text(0, 1.2, r'$\nabla^2 U = 0$', color='white', fontsize=16,
        ha='center', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig("laplacian_torus_vertical_slice_corrected.png", dpi=300)
print("Plot saved to laplacian_torus_vertical_slice_corrected.png")
plt.show()


In [ ]:
p = np.array([[R1, 0.0, 0.0]])  # point inside torus ring
Γ = model.gravity_tensor(p)
print("Trace(Γ)/(Gρ) =", np.trace(Γ)/(model.G * model.rho))


#### Example 06 Tours Acceleration Field

In [ ]:
# ===============================================================
# Example_06_Torus_Acceleration_Field.py
# ===============================================================
# Analytical Gravitational Acceleration Field of a Torus
# ===============================================================
# This example constructs a closed triangular mesh for a torus
# and computes the gravitational acceleration vectors in the
# mid-plane (z = 0) using the polyhedral gravitation model.
#
# Demonstrates:
#   • Torus mesh generation (non-star-shaped geometry)
#   • Vector field evaluation of gravitational acceleration
#   • Quiver visualization of acceleration direction and magnitude
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# ---------------------------------------------------------------
# Import the main PolyhedronGravitation module
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation

# ---------------------------------------------------------------
# 1. Torus Mesh Generation
# ---------------------------------------------------------------
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Generate a watertight triangular mesh for a torus."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2 * np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, dtype=np.int32)

# ---------------------------------------------------------------
# 2. Setup Torus Geometry and Gravity Model
# ---------------------------------------------------------------
R1, R2 = 2.0, 1.0   # Major and minor radii
a = R1 + R2 + 0.5   # Plotting domain size

vertices, faces = create_torus_tri_mesh(R1, R2, n_u=101, n_v=51)

# Non-star-shaped geometry → disable face orientation
model = PolyhedronGravitation(vertices=vertices, faces=faces,
                              density=1.0, G=1.0, orient_faces=False)
print("PolyhedronGravitation torus model initialized.")

# ---------------------------------------------------------------
# 3. Define Grid in the Plane z = 0
# ---------------------------------------------------------------
z_fixed = 0.0
x_vals = np.linspace(-a, a, 19)
y_vals = np.linspace(-a, a, 19)
X, Y = np.meshgrid(x_vals, y_vals)
grid_points = np.stack([X.ravel(), Y.ravel(),
                        np.full_like(X.ravel(), z_fixed)], axis=1)

# ---------------------------------------------------------------
# 4. Compute Gravitational Acceleration Field
# ---------------------------------------------------------------
accelerations = model.acceleration(grid_points)

# The acceleration vectors point toward the mass.
# Plot negative components for visualization of potential gradient.
U = -accelerations[:, 0].reshape(X.shape)   # g_x
V = -accelerations[:, 1].reshape(Y.shape)   # g_y

# ---------------------------------------------------------------
# 5. Plot the Quiver Field
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 8))

q = ax.quiver(X, Y, U, V, color='red', scale=80,
              pivot='middle', width=0.002, headwidth=3.5)

# --- Draw torus cross-section (two circles) ---
inner_radius = R1 - R2
outer_radius = R1 + R2
circle_inner = Circle((0, 0), inner_radius, edgecolor='black',
                      facecolor='none', linestyle='-', linewidth=1.5)
circle_outer = Circle((0, 0), outer_radius, edgecolor='black',
                      facecolor='none', linestyle='-', linewidth=1.5)
ax.add_patch(circle_inner)
ax.add_patch(circle_outer)

# --- Labels and Formatting ---
ax.set_title(r"Gravitational Acceleration Field in the Plane $z = 0$",
             fontsize=22, color='black')
ax.set_xlabel(r"$x$", fontsize=22, color='black')
ax.set_ylabel(r"$y$", fontsize=22, color='black')
ax.tick_params(axis='both', colors='black')
ax.set_xlim([-a, a])
ax.set_ylim([-a, a])
ax.set_aspect('equal')
ax.grid(False)

plt.tight_layout()
plt.savefig("gravitational_acceleration_field_torus.png",
            dpi=300, bbox_inches='tight')
plt.show()

print("Plot saved to 'gravitational_acceleration_field_torus.png'.")


#### Example 07 Torus Acceleration Vertical Slice

In [ ]:
# ===============================================================
# Example_07_Torus_Acceleration_VerticalSlice.py
# ===============================================================
# Analytical Gravitational Acceleration Field of a Torus
# in the Vertical (y–z) Plane (x = 0)
# ===============================================================
# This example constructs a closed triangular mesh for a torus
# and computes the gravitational acceleration vectors on the
# vertical plane (x = 0) using the polyhedral gravitation model.
#
# Demonstrates:
#   • Vertical-plane field evaluation
#   • Acceleration quiver visualization across cross-section
#   • Handling of non-star-shaped geometry (orient_faces = False)
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# ---------------------------------------------------------------
# Import the main PolyhedronGravitation module
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation

# ---------------------------------------------------------------
# 1. Torus Mesh Generation
# ---------------------------------------------------------------
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Generate a watertight triangular mesh for a torus."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2 * np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, dtype=np.int32)

# ---------------------------------------------------------------
# 2. Setup Torus Geometry and Gravity Model
# ---------------------------------------------------------------
R1, R2 = 2.0, 1.0
a = R1 + R2 + 0.5
vertices, faces = create_torus_tri_mesh(R1, R2, n_u=101, n_v=51)

# Non-star-shaped geometry → disable face orientation
model = PolyhedronGravitation(vertices=vertices, faces=faces,
                              density=1.0, G=1.0, orient_faces=False)
print("PolyhedronGravitation torus model initialized.")

# ---------------------------------------------------------------
# 3. Define Grid in the Vertical Plane (x = 0)
# ---------------------------------------------------------------
grid_res_y, grid_res_z = 21, 21
y_limit = R1 + R2 + 0.5
z_limit = R2 + 2.5

y_range = np.linspace(-y_limit, y_limit, grid_res_y)
z_range = np.linspace(-z_limit, z_limit, grid_res_z)
YY, ZZ = np.meshgrid(y_range, z_range)

# Evaluation points in the plane x = 0
grid_points = np.stack([np.zeros(YY.size), YY.ravel(), ZZ.ravel()], axis=1)

# ---------------------------------------------------------------
# 4. Compute Gravitational Acceleration
# ---------------------------------------------------------------
accelerations = model.acceleration(grid_points)

# Gravitational field direction (negative gradient of potential)
U = -accelerations[:, 1].reshape(YY.shape)   # g_y
V = -accelerations[:, 2].reshape(ZZ.shape)   # g_z

# ---------------------------------------------------------------
# 5. Plot the Quiver Field
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 8))

q = ax.quiver(YY, ZZ, U, V, color='red', scale=80,
              pivot='middle', width=0.002, headwidth=3.5)

# --- Draw torus boundary (two circular cross-sections) ---
circle_left = Circle((-R1, 0), R2, edgecolor='black',
                     facecolor='none', linestyle='-', linewidth=1.5)
circle_right = Circle((R1, 0), R2, edgecolor='black',
                      facecolor='none', linestyle='-', linewidth=1.5)
ax.add_patch(circle_left)
ax.add_patch(circle_right)

# --- Labels and formatting ---
ax.set_title(r"Gravitational Acceleration Field in the Plane $x = 0$",
             fontsize=22, color='black')
ax.set_xlabel(r"$y$-axis", fontsize=22, color='black')
ax.set_ylabel(r"$z$-axis", fontsize=22, color='black')
ax.tick_params(axis='both', colors='black')

# Fixed limits for visual consistency
ax.set_xlim([-3.5, 3.5])
ax.set_ylim([-3.0, 3.0])
ax.set_aspect('equal')
ax.grid(False)

plt.tight_layout()
plt.savefig("gravitational_acceleration_field_torus_vertical.png",
            dpi=300, bbox_inches='tight')
plt.show()

print("Plot saved to 'gravitational_acceleration_field_torus_vertical.png'.")


#### Example 08 - Benchmark Test

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_08_Benchmark_Python.py
# ===============================================================
# Benchmark test for PolyhedronGravitation (Python implementation)
# ===============================================================
# This script benchmarks the gravitational potential computation
# for a large polyhedron (e.g., an icosahedron) using the
# PolyhedronGravitation class in the polygrav package.
#
# It loads pre-defined geometry (vertices, faces) and evaluation
# points, computes the potential, measures elapsed time, and
# writes the results to CSV files in the data/ directory.
# ===============================================================

import numpy as np
import time
import os

# ---------------------------------------------------------------
# Import our main gravity model
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation

# ---------------------------------------------------------------
# 1. Robust Path Setup
# ---------------------------------------------------------------
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # Running interactively (no __file__): assume current directory is PolyGravitation/python
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

# ---------------------------------------------------------------
# 2. Load Geometry and Evaluation Points
# ---------------------------------------------------------------
V = np.loadtxt(os.path.join(DATA, "icosahedron_vertices.csv"), delimiter=",")
F = np.loadtxt(os.path.join(DATA, "icosahedron_faces.csv"), delimiter=",", dtype=np.int32) - 1  # subtract 1 for 0-based indexing
Pts = np.loadtxt(os.path.join(DATA, "eval_points_100k_plus_vertices.csv"), delimiter=",")

print("\n--- Python Benchmark ---")
print(f"Data folder: {DATA}")
print(f"Vertices: {V.shape[0]}, Faces: {F.shape[0]}, Points: {Pts.shape[0]}")

# ---------------------------------------------------------------
# 3. Build Model and Compute Potential
# ---------------------------------------------------------------
model = PolyhedronGravitation(vertices=V, faces=F, G=1.0, density=1.0,
                              eps=0.0, orient_faces=True)
print("PolyhedronGravitation model initialized.")

# --- timing the computation ---
t0 = time.time()
U_py = model.potential(Pts, block_size=8192)
t_py = time.time() - t0

# ---------------------------------------------------------------
# 4. Save Outputs
# ---------------------------------------------------------------
U_path = os.path.join(DATA, "U_python.csv")
T_path = os.path.join(DATA, "time_python.txt")

np.savetxt(U_path, U_py, delimiter=",")
with open(T_path, "w") as f:
    f.write(f"python_time_sec: {t_py:.6f}\n")

print(f"\nComputation complete.")
print(f"Elapsed time: {t_py:.3f} s")
print(f"Saved results to:")
print(f"  {U_path}")
print(f"  {T_path}")

# ---------------------------------------------------------------
# 5. Cleanup
# ---------------------------------------------------------------
model.close()
print("Model closed successfully.")


In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_09_Benchmark_Summary.py
# ===============================================================
# Accuracy and Performance Summary for PolyhedronGravitation
# ===============================================================
# This script compares gravitational potential results across
# implementations (Python, MATLAB, Julia Float64, Julia BigFloat)
# against a high-precision BigFloat reference (250 digits).
#
# It computes per-point runtime, mean absolute error, and RMS error,
# and outputs results to both console and summary text file.
# ===============================================================

import os, re, math
import numpy as np
from decimal import Decimal, getcontext

# ---------------------------------------------------------------
# 1. Robust Path Setup
# ---------------------------------------------------------------
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

# ---------------------------------------------------------------
# 2. Decimal Precision Context
# ---------------------------------------------------------------
getcontext().prec = 100  # sufficient for comparing 250-digit reference vs 1e-50 deltas

SUP_MINUS = "⁻"

# ---------------------------------------------------------------
# 3. Utility Functions
# ---------------------------------------------------------------
def format_sci(x, decimals=2):
    """Format like '01.49×10⁻⁴' with consistent mantissa width."""
    if not math.isfinite(x) or x == 0.0:
        return f"{'00.' + '0'*decimals}×10{SUP_MINUS}0"
    exp = int(math.floor(math.log10(abs(x))))
    mant = x / (10 ** exp)
    fmt = f"{{:0{2 + 1 + decimals}.{decimals}f}}"
    return f"{fmt.format(mant)}×10{SUP_MINUS}{abs(exp)}"

def read_time(path):
    """Extract first numeric time value (seconds) from text file."""
    if not os.path.exists(path):
        return np.nan
    with open(path) as f:
        s = f.read()
    m = re.search(r'([0-9]*\.?[0-9]+)', s)
    return float(m.group(1)) if m else np.nan

# ---------------------------------------------------------------
# 4. Load High-Precision Reference
# ---------------------------------------------------------------
def load_reference_dec():
    """Load BigFloat reference potential values (250-digit precision)."""
    txt_ref = os.path.join(DATA, "U_ref_bigfloat_250digits.txt")
    csv_ref = os.path.join(DATA, "U_ref_bigfloat.csv")
    if os.path.exists(txt_ref):
        with open(txt_ref) as f:
            ref = [Decimal(line.strip()) for line in f if line.strip()]
        return ref
    elif os.path.exists(csv_ref):
        arr = np.loadtxt(csv_ref, delimiter=",")
        return [Decimal(str(v)) for v in arr.tolist()]
    else:
        raise FileNotFoundError("Missing reference file: expected 'U_ref_bigfloat_250digits.txt' or 'U_ref_bigfloat.csv'.")

U_ref_dec = load_reference_dec()
Nref = len(U_ref_dec)

# ---------------------------------------------------------------
# 5. Data Loading Helpers
# ---------------------------------------------------------------
def load_decimals_from_txt(path):
    """Load one value per line as Decimal."""
    with open(path) as f:
        return [Decimal(line.strip()) for line in f if line.strip()]

def load_float_csv_as_decimal(path):
    """Load CSV float64 values, convert to Decimal (via string for safety)."""
    arr = np.loadtxt(path, delimiter=",", dtype=np.float64)
    return [Decimal(str(v)) for v in arr.tolist()]

def compute_row(code, ufile, tfile, as_decimal_file=False):
    """
    Compute runtime per point, mean error, and RMS error.
    If as_decimal_file=True, read ufile as text of Decimals (Julia BigFloat output).
    """
    u_path = os.path.join(DATA, ufile)
    t_path = os.path.join(DATA, tfile)
    if not os.path.exists(u_path):
        return None

    if as_decimal_file:
        U_dec = load_decimals_from_txt(u_path)
    else:
        U_dec = load_float_csv_as_decimal(u_path)

    n = min(len(U_dec), Nref)
    if n == 0:
        return None

    abs_sum = Decimal(0)
    sq_sum = Decimal(0)
    for i in range(n):
        d = (U_ref_dec[i] - U_dec[i]).copy_abs()
        abs_sum += d
        sq_sum += d * d

    mean_abs = abs_sum / Decimal(n)
    rms_abs = (sq_sum / Decimal(n)).sqrt()

    # Time per point
    t = read_time(t_path)
    tpp = (t / n) if np.isfinite(t) and n > 0 else np.nan

    return code, tpp, float(mean_abs), float(rms_abs)

# ---------------------------------------------------------------
# 6. Compute All Rows
# ---------------------------------------------------------------
rows = [
    compute_row("Python",   "U_python.csv",                   "time_python.txt",              as_decimal_file=False),
    compute_row("MATLAB",   "U_matlab_parallel.csv",          "time_matlab_parallel.txt",     as_decimal_file=False),
    compute_row("Julia_MP", "U_julia_float64_fast.csv",       "time_julia_float64_fast.txt",  as_decimal_file=False),
    compute_row("Julia50",  "U_julia_bigfloat_50.csv",        "time_julia_bigfloat_50.txt",   as_decimal_file=True),
]
rows = [r for r in rows if r is not None]

# ---------------------------------------------------------------
# 7. Print and Save Summary
# ---------------------------------------------------------------
header_title = "=== Gravitational Potential Accuracy & Speed Summary ==="
header_ref = "Reference: Julia BigFloat (250 digits)"
header_note = "Julia 50 comparison performed at 52-digit arithmetic precision."

print(header_title)
print(header_ref)
print(header_note)
print("\nCode      runtime (s/pt)     mean(error)       RMS(error)")

for i, (name, tpp, mean, rms) in enumerate(rows):
    print(f"{i:<5}{name:10s} {format_sci(tpp,2):>16}  {format_sci(mean,2):>14}  {format_sci(rms,2):>14}")

out_path = os.path.join(DATA, "summary_runtime_accuracy.txt")
with open(out_path, "w") as f:
    f.write(header_title + "\n")
    f.write(header_ref + "\n")
    f.write(header_note + "\n\n")
    f.write("Code      runtime (s/pt)     mean(error)       RMS(error)\n")
    for i, (name, tpp, mean, rms) in enumerate(rows):
        f.write(f"{i:<5}{name:10s} {format_sci(tpp,2):>16}  {format_sci(mean,2):>14}  {format_sci(rms,2):>14}\n")

print(f"\nSaved summary to: {out_path}")


#### Example 10 Tours Gravitational Potential 

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib import cm
from matplotlib.colors import ListedColormap

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_10_Torus_ZeroG_CircleFit.py
# ===============================================================
# Gravitational Potential and Zero-Acceleration Locus (z = 0 plane)
# ===============================================================
# This example computes the gravitational potential of a torus
# and identifies the zero-acceleration curve (where gₓ = gᵧ = 0)
# in the mid-plane. The zero-g points are then used to fit and
# visualize a circular locus representing equilibrium points.
#
# Demonstrates:
#   • Combined potential + acceleration field computation
#   • Detection of zero-g regions via sign change logic
#   • Circle fit through zero-acceleration points
#   • Use of custom MATLAB-style “ajet” colormap
# ===============================================================

# ---------------------------------------------------------------
# Import your analytical polyhedron gravity model
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation


# ===============================================================
# 0. Utility: ajet colormap (MATLAB-style jet)
# ===============================================================
def ajet(n=64):
    """Generate softened jet colormap equivalent to MATLAB’s ajet."""
    if not (isinstance(n, int) and n > 0):
        raise ValueError("Input argument n must be a positive integer")
    m = n // 6
    cm_jet = cm.get_cmap("jet", n + 2 * m)
    colors = cm_jet(np.linspace(0, 1, n + 2 * m))
    colors = colors[m:-m]
    return ListedColormap(colors, name="ajet")


# ===============================================================
# 1. Torus Mesh Generation
# ===============================================================
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Generate a watertight triangular mesh for a torus."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2 * np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, dtype=np.int32)


# ===============================================================
# 2. Setup Torus Geometry and Gravity Model
# ===============================================================
R1, R2 = 2.0, 1.0  # major & minor radii
a = R1 + R2 + 0.5  # plotting domain size
vertices, faces = create_torus_tri_mesh(R1, R2, n_u=101, n_v=51)

# Instantiate PolyhedronGravitation (no face orientation for torus)
model = PolyhedronGravitation(vertices=vertices, faces=faces,
                              density=1.0, G=1.0, orient_faces=False)
print("PolyhedronGravitation torus model initialized.")


# ===============================================================
# 3. Define Grid in Plane z = 0
# ===============================================================
z_fixed = 0.0
x_vals = np.linspace(-a, a, 101)
y_vals = np.linspace(-a, a, 101)
X, Y = np.meshgrid(x_vals, y_vals)
grid_points = np.stack([X.ravel(), Y.ravel(),
                        np.full_like(X.ravel(), z_fixed)], axis=1)


# ===============================================================
# 4. Compute Potential and Acceleration
# ===============================================================
potentials = model.potential(grid_points)
Phi = potentials.reshape(X.shape)

accelerations = model.acceleration(grid_points)
Ax = accelerations[:, 0].reshape(X.shape)
Ay = accelerations[:, 1].reshape(Y.shape)

# --- Detect zero-crossing cells for Ax and Ay ---
def has_zero_in_cell(A00, A10, A01, A11):
    cell_min = np.minimum.reduce([A00, A10, A01, A11])
    cell_max = np.maximum.reduce([A00, A10, A01, A11])
    return (cell_min <= 0) & (cell_max >= 0)

Ax00, Ax10, Ax01, Ax11 = Ax[:-1, :-1], Ax[1:, :-1], Ax[:-1, 1:], Ax[1:, 1:]
Ay00, Ay10, Ay01, Ay11 = Ay[:-1, :-1], Ay[1:, :-1], Ay[:-1, 1:], Ay[1:, 1:]
mask_Ax = has_zero_in_cell(Ax00, Ax10, Ax01, Ax11)
mask_Ay = has_zero_in_cell(Ay00, Ay10, Ay01, Ay11)
cell_mask = mask_Ax & mask_Ay

# Candidate zero-g cell centers
Xc = 0.25 * (X[:-1, :-1] + X[1:, :-1] + X[:-1, 1:] + X[1:, 1:])
Yc = 0.25 * (Y[:-1, :-1] + Y[1:, :-1] + Y[:-1, 1:] + Y[1:, 1:])
cand_x = Xc[cell_mask]
cand_y = Yc[cell_mask]

# Cluster nearby duplicates
dx = x_vals[1] - x_vals[0]
dy = y_vals[1] - y_vals[0]
merge_radius2 = (1.5 * max(dx, dy))**2
x_zeros, y_zeros = [], []
for x0, y0 in zip(cand_x, cand_y):
    if all((x0 - x1)**2 + (y0 - y1)**2 > merge_radius2 for x1, y1 in zip(x_zeros, y_zeros)):
        x_zeros.append(x0)
        y_zeros.append(y0)

x_zeros = np.array(x_zeros)
y_zeros = np.array(y_zeros)

# --- Symmetry-based circle estimate ---
R = np.mean(np.sqrt(x_zeros**2 + y_zeros**2))
print(f"Symmetry-enforced zero-g circle radius = {R:.4f}")

# Points along fitted circle
theta = np.linspace(0, 2*np.pi, 100)
circle_x = R * np.cos(theta)
circle_y = R * np.sin(theta)


# ===============================================================
# 5. Plot the Potential Field (Contour + Zero-g Circle)
# ===============================================================
fig, ax = plt.subplots(figsize=(9, 8))
cmap_custom = ajet(64)

c = ax.contourf(X, Y, Phi, levels=30, cmap=cmap_custom)
plt.colorbar(c, ax=ax, label="Gravitational Potential", fraction=0.046, pad=0.04)

# --- Torus cross-section (two circles) ---
inner_radius = R1 - R2
outer_radius = R1 + R2
ax.add_patch(Circle((0, 0), inner_radius, edgecolor='black',
                    facecolor='none', linestyle='-', linewidth=1.5))
ax.add_patch(Circle((0, 0), outer_radius, edgecolor='black',
                    facecolor='none', linestyle='-', linewidth=1.5))

# --- Zero-acceleration locus ---
ax.plot(circle_x, circle_y, 'k.', markersize=4, label="Zero-acceleration locus")
ax.plot(0, 0, 'ko', markersize=3)  # black dot at origin

# --- Labels and Formatting ---
ax.set_title(r"Gravitational Potential and Zero-$g$ Locus (Plane $z=0$)",
             fontsize=22, color='black')
ax.set_xlabel(r"$x$-axis", fontsize=20, color='black')
ax.set_ylabel(r"$y$-axis", fontsize=20, color='black')
ax.tick_params(axis='both', colors='black')
ax.set_xlim([-a, a])
ax.set_ylim([-a, a])
ax.set_aspect('equal')
ax.grid(False)
ax.legend(fontsize=12, loc="lower right")

plt.tight_layout()
plt.savefig("gravitational_potential_field_torus_zeroG.png",
            dpi=300, bbox_inches='tight')
plt.show()

print("Plot saved to 'gravitational_potential_field_torus_zeroG.png'.")


#### Example 11 Tours Gravitational Potential Vertical Plane 

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_11_Torus_ZeroG_VerticalPlane.py
# ===============================================================
# Gravitational Potential and Zero-Acceleration Points in x = 0 Plane
# ===============================================================
# This example computes the gravitational potential and acceleration
# for a torus in the vertical (y–z) plane, detects regions where both
# a_y and a_z change sign (zero-acceleration points), and visualizes
# these equilibrium points on top of the potential field contours.
#
# Demonstrates:
#   • Detection of equilibrium points in cross-section
#   • Zero-crossing detection using sign logic
#   • Use of custom “ajet” colormap
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib import cm
from matplotlib.colors import ListedColormap

# ---------------------------------------------------------------
# Import analytical polyhedron gravity model
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation


# ===============================================================
# 0. Utility — MATLAB-style “ajet” colormap
# ===============================================================
def ajet(n=64):
    """Generate softened jet colormap (MATLAB ajet equivalent)."""
    if not (isinstance(n, int) and n > 0):
        raise ValueError("Input argument n must be a positive integer")
    m = n // 6
    cm_jet = cm.get_cmap("jet", n + 2 * m)
    colors = cm_jet(np.linspace(0, 1, n + 2 * m))
    colors = colors[m:-m]
    return ListedColormap(colors, name="ajet")


# ===============================================================
# 1. Torus Mesh Generation
# ===============================================================
def create_torus_tri_mesh(R1, R2, n_u=32, n_v=16):
    """Create watertight triangular mesh for a torus."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, 2 * np.pi, n_v, endpoint=False)
    u_grid, v_grid = np.meshgrid(u, v)
    X = (R1 + R2 * np.cos(v_grid)) * np.cos(u_grid)
    Y = (R1 + R2 * np.cos(v_grid)) * np.sin(u_grid)
    Z = R2 * np.sin(v_grid)
    V = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

    faces = []
    for i in range(n_v):
        for j in range(n_u):
            v0, v1 = i * n_u + j, i * n_u + (j + 1) % n_u
            v2, v3 = ((i + 1) % n_v) * n_u + j, ((i + 1) % n_v) * n_u + (j + 1) % n_u
            faces.append([v0, v2, v1])
            faces.append([v1, v2, v3])
    return V, np.array(faces, dtype=np.int32)


# ===============================================================
# 2. Setup Geometry and Gravity Model
# ===============================================================
R1, R2 = 2.0, 1.0  # major and minor radii
vertices, faces = create_torus_tri_mesh(R1, R2, n_u=101, n_v=51)

model = PolyhedronGravitation(vertices=vertices, faces=faces,
                              density=1.0, G=1.0, orient_faces=False)
print("PolyhedronGravitation torus model initialized.")

# ===============================================================
# 3. Define Vertical (x = 0) Grid
# ===============================================================
grid_res_y, grid_res_z = 201, 201
y_limit = R1 + R2 + 0.8
z_limit = R2 + 2.5

y_range = np.linspace(-y_limit, y_limit, grid_res_y)
z_range = np.linspace(-z_limit, z_limit, grid_res_z)
YY, ZZ = np.meshgrid(y_range, z_range)

grid_points = np.stack([np.zeros(YY.size), YY.ravel(), ZZ.ravel()], axis=1)


# ===============================================================
# 4. Compute Potential and Acceleration
# ===============================================================
print(f"Computing potential and acceleration on {grid_res_y}×{grid_res_z} grid (x=0 plane)...")

potentials = model.potential(grid_points)
Phi = potentials.reshape(YY.shape)

accelerations = model.acceleration(grid_points)
Ay = accelerations[:, 1].reshape(YY.shape)  # a_y
Az = accelerations[:, 2].reshape(ZZ.shape)  # a_z


# --- detect zero-crossing cells for Ay and Az ---
def has_zero_in_cell(A00, A10, A01, A11):
    """Return True for cells where the value crosses zero."""
    cell_min = np.minimum.reduce([A00, A10, A01, A11])
    cell_max = np.maximum.reduce([A00, A10, A01, A11])
    return (cell_min <= 0) & (cell_max >= 0)


Ay00, Ay10, Ay01, Ay11 = Ay[:-1, :-1], Ay[1:, :-1], Ay[:-1, 1:], Ay[1:, 1:]
Az00, Az10, Az01, Az11 = Az[:-1, :-1], Az[1:, :-1], Az[:-1, 1:], Az[1:, 1:]
mask_Ay = has_zero_in_cell(Ay00, Ay10, Ay01, Ay11)
mask_Az = has_zero_in_cell(Az00, Az10, Az01, Az11)
cell_mask = mask_Ay & mask_Az

# --- approximate zero-g cell centers ---
Yc = 0.25 * (YY[:-1, :-1] + YY[1:, :-1] + YY[:-1, 1:] + YY[1:, 1:])
Zc = 0.25 * (ZZ[:-1, :-1] + ZZ[1:, :-1] + ZZ[:-1, 1:] + ZZ[1:, 1:])
cand_y = Yc[cell_mask]
cand_z = Zc[cell_mask]

# --- cluster duplicates ---
dy = y_range[1] - y_range[0]
dz = z_range[1] - z_range[0]
merge_radius2 = (1.5 * max(dy, dz))**2

y_zeros, z_zeros = [], []
for y0, z0 in zip(cand_y, cand_z):
    if all((y0 - y1)**2 + (z0 - z1)**2 > merge_radius2 for y1, z1 in zip(y_zeros, z_zeros)):
        y_zeros.append(y0)
        z_zeros.append(z0)

y_zeros = np.array(y_zeros)
z_zeros = np.array(z_zeros)

print("\nZero-acceleration equilibrium points (approx):")
for y, z in zip(y_zeros, z_zeros):
    print(f"  (y = {y:.4f}, z = {z:.4f})")


# ===============================================================
# 5. Plot Potential Field with Equilibrium Points
# ===============================================================
fig, ax = plt.subplots(figsize=(9, 8))
cmap_custom = ajet(64)

# Filled potential contours
c = ax.contourf(YY, ZZ, Phi, levels=30, cmap=cmap_custom)
plt.colorbar(c, ax=ax, label="Gravitational Potential", fraction=0.046, pad=0.04)

# --- Torus vertical cross-section (two circles) ---
ax.add_patch(Circle((-R1, 0), R2, edgecolor='black', facecolor='none', linewidth=1.2))
ax.add_patch(Circle((R1, 0), R2, edgecolor='black', facecolor='none', linewidth=1.2))

# --- zero-g points ---
ax.scatter(y_zeros, z_zeros, s=12, c='black', marker='o', label="Zero-acceleration points")

# --- formatting ---
ax.set_title(r"Gravitational Potential and Zero-$g$ Points (Plane $x=0$)",
             fontsize=22, color='black')
ax.set_xlabel(r"$y$-axis", fontsize=20, color='black')
ax.set_ylabel(r"$z$-axis", fontsize=20, color='black')
ax.tick_params(axis='both', colors='black')
ax.set_xlim([-y_limit, y_limit])
ax.set_ylim([-z_limit, z_limit])
ax.set_aspect('equal')
ax.grid(False)
ax.legend(fontsize=12, loc="lower right")

plt.tight_layout()
plt.savefig("gravitational_potential_field_torus_vertical_zeroG.png",
            dpi=300, bbox_inches='tight')
plt.show()

print("\nPlot saved to 'gravitational_potential_field_torus_vertical_zeroG.png'.")


#### Example 12 Icosahedron Precision Bench Mark Test

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_12_Icosahedron_Face_PrecisionTest.py
# ===============================================================
# Analytical Polyhedron Gravity – Edge & Face Perturbation Test
# ===============================================================
# This example constructs a transformed icosahedron where one face
# lies on integer coordinates (0,0,0), (1,1,0), (1,0,1), then evaluates
# the potential, acceleration, and gravity tensor at:
#   • Vertices, edges, and face centers
#   • Slightly shifted ("ε-perturbed") versions of each point
#
# Demonstrates:
#   • Fractional coordinate formatting
#   • Precision sensitivity near edges and faces
#   • Componentwise output of gravity tensor
# ===============================================================

import numpy as np
from fractions import Fraction

# ---------------------------------------------------------------
# 0. Import gravity model (fallback placeholder if missing)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: The 'polygrav' module is not installed or accessible.")
    print("Using placeholder class; results will be mock data.\n")

    class PolyhedronGravitation:
        def __init__(self, **kwargs):
            pass
        def potential(self, point): return 0.0
        def acceleration(self, point): return np.zeros(3)
        def gravity_tensor(self, point): return np.zeros((3, 3))


# ===============================================================
# 1. Helper Functions
# ===============================================================
def normalize(v: np.ndarray) -> np.ndarray:
    """Normalize a vector to unit length."""
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def format_point_as_fraction(point: np.ndarray) -> str:
    """Format 3D coordinates as integer or fractional strings."""
    coords = []
    for c in point:
        frac = Fraction(c).limit_denominator(1_000_000)
        coords.append(str(frac.numerator) if frac.denominator == 1
                      else f"{frac.numerator}/{frac.denominator}")
    return f"({coords[0]}, {coords[1]}, {coords[2]})"

def format_point_as_float(point: np.ndarray) -> str:
    """Format 3D coordinates as floating-point string."""
    return f"({point[0]:.10e}, {point[1]:.10e}, {point[2]:.10e})"


# ===============================================================
# 2. Create Transformed Icosahedron
# ===============================================================
def create_transformed_icosahedron():
    """
    Create an icosahedron where one face lies on integer coordinates
    (0,0,0), (1,1,0), (1,0,1). Returns vertices (V) and faces (F).
    """
    φ = (1.0 + np.sqrt(5.0)) / 2.0
    V_std = np.array([
        [-1,  φ, 0], [ 1,  φ, 0], [-1, -φ, 0], [ 1, -φ, 0],
        [0, -1,  φ], [0,  1,  φ], [0, -1, -φ], [0,  1, -φ],
        [ φ, 0, -1], [ φ, 0,  1], [-φ, 0, -1], [-φ, 0,  1]
    ], dtype=float)

    F = np.array([
        [0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11],
        [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6], [7, 1, 8],
        [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9],
        [4, 9, 5], [2, 4, 11], [6, 2, 10], [8, 6, 7], [9, 8, 1]
    ], dtype=np.int32)

    # Source face (from standard icosahedron)
    s1, s2, s3 = V_std[[0, 11, 5]]
    # Target face with integer coordinates
    t1, t2, t3 = np.array([0., 0., 0.]), np.array([1., 1., 0.]), np.array([1., 0., 1.])

    # Construct rotation/scale matrices
    scale = np.linalg.norm(t2 - t1) / np.linalg.norm(s2 - s1)
    u_s = normalize(s2 - s1)
    w_s = normalize(np.cross(u_s, s3 - s1))
    v_s = np.cross(w_s, u_s)
    A = np.c_[u_s, v_s, w_s]

    u_t = normalize(t2 - t1)
    w_t = normalize(np.cross(u_t, t3 - t1))
    v_t = np.cross(w_t, u_t)
    B = np.c_[u_t, v_t, w_t]

    R = B @ A.T  # rotation
    V_trans = np.array([((R @ (p - s1)) * scale) + t1 for p in V_std])
    return V_trans, F


# ===============================================================
# 3. Main Execution
# ===============================================================
if __name__ == "__main__":
    G, density, ε = 1.0, 1.0, 1e-10

    V, F = create_transformed_icosahedron()
    print(f"Generated an icosahedron with {V.shape[0]} vertices and {F.shape[0]} faces.")
    print("One face is defined by integer coordinates: (0,0,0), (1,1,0), (1,0,1).\n")

    # Define test points
    v1, v2, v3 = np.array([0,0,0]), np.array([1,1,0]), np.array([1,0,1])
    face_center = (v1 + v2 + v3) / 3.0
    test_points = {
        "Vertex": v2,
        "On Edge": (v1 + v2) / 2.0,
        "On Face": face_center,
        "On Extended Edge": v1 + 2.0 * (v2 - v1),
        "On Extended Face": face_center + np.array([1.0, 0, 0]),
        "Interior": face_center / 2.0,
        "Exterior": np.array([2.0, 2.0, 2.0]),
    }

    model = PolyhedronGravitation(vertices=V, faces=F, G=G, density=density, orient_faces=True)

    # -----------------------------------------------------------
    # Output Header
    # -----------------------------------------------------------
    header = (f"{'Point ID':<25s} | {'Test Point (x,y,z)':<35s} | {'Potential':<25s} | "
              f"{'Acceleration (gx,gy,gz)':<65s} | {'Tensor (xx,xy,xz,yy,yz,zz)':<100s}")
    print("-" * len(header))
    print(header)
    print("-" * len(header))

    # -----------------------------------------------------------
    # Evaluate Each Point (Original + Shifted)
    # -----------------------------------------------------------
    for name, p_orig in test_points.items():
        # Compute ε-shift
        if name == "On Edge":
            shift_dir = normalize(v2 - v1)
        elif name == "On Face":
            shift_dir = normalize(v2 - v1)
        else:
            shift_dir = np.ones(3)

        p_shift = p_orig + ε * shift_dir

        cases = [
            (p_orig, f"{name} (original)", format_point_as_fraction),
            (p_shift, f"{name} (shifted)", format_point_as_float),
        ]

        for p, label, fmt_fn in cases:
            try:
                U = model.potential(p)
                g = model.acceleration(p)
                Γ = model.gravity_tensor(p)

                # Tensor components
                g_xx, g_xy, g_xz = Γ[0, 0], Γ[0, 1], Γ[0, 2]
                g_yy, g_yz, g_zz = Γ[1, 1], Γ[1, 2], Γ[2, 2]

                print(f"{label:<25s} | {fmt_fn(p):<35s} | "
                      f"{U:>+1.14e} | "
                      f"({g[0]:+1.14e}, {g[1]:+1.14e}, {g[2]:+1.14e}) "
                      f"| ({g_xx:+1.14e}, {g_xy:+1.14e}, {g_xz:+1.14e}, "
                      f"{g_yy:+1.14e}, {g_yz:+1.14e}, {g_zz:+1.14e})")
            except Exception as e:
                print(f"{label:<25s} | {fmt_fn(p):<35s} | Calculation failed: {e}")

    print("-" * len(header))
    print("\nComputation complete.")


In [ ]:
# --- Matplotlib plotting style setup ---
# This configuration ensures the plots have a professional, publication-quality look.
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False, # Set to False for broader compatibility, True if LaTeX is installed
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman"],
    "axes.labelsize": 20,
    "font.size": 20,
    "legend.fontsize": 14,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

In [ ]:
import psutil, os, time

def measure_threads(func, *args, **kwargs):
    proc = psutil.Process(os.getpid())
    before = len(proc.threads())
    t0 = time.perf_counter()
    func(*args, **kwargs)
    dt = time.perf_counter() - t0
    after = len(proc.threads())
    print(f"Time: {dt:.3f} s | Threads before: {before}, after: {after}")

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_13_SpeedComparison_PolyhedronModels.py
# ===============================================================
# SPEED COMPARISON: Periyandy & Bevis (vectorized) vs Werner–Arribas (legacy)
# ===============================================================
# This benchmark compares the computational performance of:
#   • Periyandy & Bevis (2025) analytical polyhedron gravity model
#   • Werner–Arribas (legacy numerical approach, non-vectorized)
#   • Finite-difference numerical approximations for validation
#
# Metrics: Runtime vs number of observation points (log–log scale)
# ===============================================================

import numpy as np
import time
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# 0. Import new and legacy gravity models
# ---------------------------------------------------------------
from polygravitation import PolyhedronGravitation
from Polyhedron_WS.GP_Polyhedron_WS import Polyhedron as Polyhedron_WS


# ===============================================================
# 1. Numerical Helper Functions (Finite Differences)
# ===============================================================
def g_num_class(model, points):
    """Compute numerical acceleration via finite differences."""
    h = 1e-7
    accelerations = np.zeros_like(points)
    for i, p in enumerate(points):
        x, y, z = p
        φ = lambda pt: model.potential(pt)
        gx = -(φ([x + h, y, z]) - φ([x - h, y, z])) / (2 * h)
        gy = -(φ([x, y + h, z]) - φ([x, y - h, z])) / (2 * h)
        gz = -(φ([x, y, z + h]) - φ([x, y, z - h])) / (2 * h)
        accelerations[i, :] = [gx, gy, gz]
    return accelerations


def tensor_num_class(model, points):
    """Compute numerical gravity tensor via finite differences."""
    h = 1e-7
    tensors = np.zeros((points.shape[0], 3, 3))
    for i, p in enumerate(points):
        x, y, z = p
        g_xp = model.acceleration([x + h, y, z])
        g_xm = model.acceleration([x - h, y, z])
        g_yp = model.acceleration([x, y + h, z])
        g_ym = model.acceleration([x, y - h, z])
        g_zp = model.acceleration([x, y, z + h])
        g_zm = model.acceleration([x, y, z - h])
        col_x = (g_xp - g_xm) / (2 * h)
        col_y = (g_yp - g_ym) / (2 * h)
        col_z = (g_zp - g_zm) / (2 * h)
        tensors[i, :, :] = np.array([col_x, col_y, col_z]).T
    return tensors


# ===============================================================
# 2. Geometry Setup — Unit Cube
# ===============================================================
vertices = np.array([
    [-0.5, -0.5, -0.5], [ 0.5, -0.5, -0.5], [ 0.5,  0.5, -0.5], [-0.5,  0.5, -0.5],
    [-0.5, -0.5,  0.5], [ 0.5, -0.5,  0.5], [ 0.5,  0.5,  0.5], [-0.5,  0.5,  0.5]
])
faces = np.array([
    [0, 3, 2], [0, 2, 1], [4, 5, 6], [4, 6, 7],
    [0, 1, 5], [0, 5, 4], [2, 3, 7], [2, 7, 6],
    [0, 4, 7], [0, 7, 3], [1, 2, 6], [1, 6, 5]
], dtype=np.int32)

# Instantiate analytical models
model_pb = PolyhedronGravitation(vertices=vertices, faces=faces,
                                 density=1.0, G=1.0, orient_faces=True)
model_ws = Polyhedron_WS(vertices=vertices, face_indexes=faces, density=1.0)

print("Initialized analytical models:")
print(" • Periyandy & Bevis (vectorized)")
print(" • Werner–Arribas (legacy, non-vectorized)\n")


# ===============================================================
# 3. Speed Benchmark Setup
# ===============================================================
n_points_range = np.logspace(1, 5, 201, dtype=int)   # 10 → 10⁵
results = {
    "Periyandy & Bevis": {"Potential": [], "Acceleration": [], "Tensor": []},
    "Werner–Arribas": {"Potential": [], "Acceleration": [], "Tensor": []},
    "Numerical": {"Acceleration": [], "Tensor": []}
}
numerical_n_accel = []
numerical_n_tensor = []


print("--- Starting Speed Benchmark ---")
for n in n_points_range:
    points = np.random.rand(n, 3) * 5.0 + 2.0  # Outside cube
    # print(f"  Evaluating {n} points...")

    # --- Periyandy & Bevis (vectorized) ---
    t0 = time.time()
    model_pb.potential(points)
    results["Periyandy & Bevis"]["Potential"].append(time.time() - t0)

    t0 = time.time()
    model_pb.acceleration(points)
    results["Periyandy & Bevis"]["Acceleration"].append(time.time() - t0)

    t0 = time.time()
    model_pb.gravity_tensor(points)
    results["Periyandy & Bevis"]["Tensor"].append(time.time() - t0)

    # --- Werner–Arribas (looped) ---
    t0 = time.time()
    for p in points:
        model_ws.U(p)
    results["Werner–Arribas"]["Potential"].append(time.time() - t0)

    t0 = time.time()
    for p in points:
        model_ws.g(p)
    results["Werner–Arribas"]["Acceleration"].append(time.time() - t0)

    t0 = time.time()
    for p in points:
        model_ws.gravity_gradients(p)
    results["Werner–Arribas"]["Tensor"].append(time.time() - t0)

    # --- Numerical (subset only, for cost reasons) ---
    if n <= 1000:
        t0 = time.time()
        g_num_class(model_pb, points)
        results["Numerical"]["Acceleration"].append(time.time() - t0)
        numerical_n_accel.append(n)

    if n <= 100:
        t0 = time.time()
        tensor_num_class(model_pb, points)
        results["Numerical"]["Tensor"].append(time.time() - t0)
        numerical_n_tensor.append(n)

print("--- Benchmark Complete ---\n")


# ===============================================================
# 4. Plot Results (3×1 Subplots)
# ===============================================================
fig, axes = plt.subplots(3, 1, figsize=(8, 12), sharex=True)

pb_color = np.array([0, 174, 235]) / 255   # Periyandy & Bevis blue
wa_color = "green"                         # Werner–Arribas green
num_color = "red"                          # Numerical red
marker_size = 3

# --- Potential ---
ax = axes[0]
ax.loglog(n_points_range, results["Periyandy & Bevis"]["Potential"], 'o',
          color=pb_color, markersize=marker_size, label="Periyandy & Bevis")
ax.loglog(n_points_range, results["Werner–Arribas"]["Potential"], 'o',
          color=wa_color, markersize=marker_size, label="Werner–Arribas")
ax.set_ylabel("Time (s)")
ax.set_title("Potential Computation Speed")
ax.grid(True, which="both", ls="--")
ax.legend(loc="lower right")

# --- Acceleration ---
ax = axes[1]
ax.loglog(n_points_range, results["Periyandy & Bevis"]["Acceleration"], 'o',
          color=pb_color, markersize=marker_size, label="Periyandy & Bevis")
ax.loglog(n_points_range, results["Werner–Arribas"]["Acceleration"], 'o',
          color=wa_color, markersize=marker_size, label="Werner–Arribas")
ax.loglog(numerical_n_accel, results["Numerical"]["Acceleration"], 'o',
          color=num_color, markersize=marker_size, label="Numerical FD")
ax.set_ylabel("Time (s)")
ax.set_title("Acceleration Computation Speed")
ax.grid(True, which="both", ls="--")
ax.legend(loc="lower right")

# --- Tensor ---
ax = axes[2]
ax.loglog(n_points_range, results["Periyandy & Bevis"]["Tensor"], 'o',
          color=pb_color, markersize=marker_size, label="Periyandy & Bevis")
ax.loglog(n_points_range, results["Werner–Arribas"]["Tensor"], 'o',
          color=wa_color, markersize=marker_size, label="Werner–Arribas")
ax.loglog(numerical_n_tensor, results["Numerical"]["Tensor"], 'o',
          color=num_color, markersize=marker_size, label="Numerical FD")
ax.set_xlabel("Number of Observation Points")
ax.set_ylabel("Time (s)")
ax.set_title("Tensor Computation Speed")
ax.grid(True, which="both", ls="--")
ax.legend(loc="lower right")

fig.tight_layout()
fig.savefig("gravity_speed_test_subplots.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("Plot saved to 'gravity_speed_test_subplots.pdf'.")


In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_12_Cube_Laplacian_SurfaceTest.py
# ===============================================================
# Analytical Polyhedron Gravity – Cube Laplacian Surface Test
# ===============================================================

import numpy as np
from math import pi

# ---------------------------------------------------------------
# 0. Import gravity model (fallback placeholder)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: 'polygravitation' not installed. Using mock model.\n")
    class PolyhedronGravitation:
        def __init__(self, **kwargs): pass
        def potential(self, point): return 0.0
        def acceleration(self, point): return np.zeros(3)
        def gravity_tensor(self, point): return np.zeros((3, 3))


# ===============================================================
# 1. Cube Geometry
# ===============================================================
def create_cube_vertices_faces():
    V = np.array([
        [0.,0.,0.], [2.,0.,0.], [2.,2.,0.], [0.,2.,0.],
        [0.,0.,2.], [2.,0.,2.], [2.,2.,2.], [0.,2.,2.]
    ])

    F = np.array([
        [0,1,2],[0,2,3],       # bottom
        [4,6,5],[4,7,6],       # top
        [0,5,1],[0,4,5],       # front
        [3,2,6],[3,6,7],       # back
        [0,3,7],[0,7,4],       # left
        [1,5,6],[1,6,2],       # right
    ])

    return V, F


def fmt(p):
    """Format with integer if exact, else scientific."""
    if np.all(np.isclose(p, np.round(p))):
        p = np.round(p).astype(int)
        return f"({p[0]}, {p[1]}, {p[2]})"
    return f"({p[0]:.15e}, {p[1]:.15e}, {p[2]:.15e})"


# ===============================================================
# 2. Main Execution
# ===============================================================
if __name__ == "__main__":
    G = 1.0
    rho = 1.0
    eps = 1e-10

    V, F = create_cube_vertices_faces()

    model = PolyhedronGravitation(vertices=V, faces=F,
                                  G=G, density=rho, orient_faces=True)

    # -----------------------------------------------------------
    # Test Points (exactly as requested)
    # -----------------------------------------------------------
    test_points = {
        # === Vertex Points ===
        "V1 original": np.array([2., 0., 2.]),
        "V1 exterior": np.array([2.+eps, 0.+eps, 2.+eps]),
        "V1 interior": np.array([2.-eps, 0.+eps, 2.-eps]),

        "V2 original": np.array([2., 2., 2.]),
        "V2 exterior": np.array([2.+eps, 2.+eps, 2.+eps]),
        "V2 interior": np.array([2.-eps, 2.-eps, 2.-eps]),

        # === Face Points ===
        "F1 original": np.array([1., 1., 2.]),
        "F1 interior": np.array([1., 1., 2.-eps]),
        "F1 exterior": np.array([1., 1., 2.+eps]),

        "F2 original": np.array([2., 1., 1.]),
        "F2 interior": np.array([2.-eps, 1., 1.]),
        "F2 exterior": np.array([2.+eps, 1., 1.]),
    }

    # -----------------------------------------------------------
    # Table Output
    # -----------------------------------------------------------
    header = f"{'Point ID':<18s} | {'Test Point (x,y,z)':<40s} | {'Laplacian L':>24s}"
    sep = "-" * len(header)

    print(sep)
    print(header)
    print(sep)

    lines = [sep, header, sep]

    for name, p in test_points.items():
        try:
            Γ = model.gravity_tensor(p)
            L = np.trace(Γ) / (4 * pi * G * rho)

            line = f"{name:<18s} | {fmt(p):<40s} | {L:+.15e}"
            print(line)
            lines.append(line)

        except Exception as e:
            line = f"{name:<18s} | {fmt(p):<40s} | FAILED: {e}"
            print(line)
            lines.append(line)

    print(sep)
    lines.append(sep)
    lines.append("Computation complete.")

    # -----------------------------------------------------------
    # Save to file
    # -----------------------------------------------------------
    fname = "Example_12_Cube_Laplacian_SurfaceTest.txt"
    with open(fname, "w") as f:
        for L in lines:
            f.write(L + "\n")

    print(f"\nSaved results to '{fname}'.")


In [2]:
#!/usr/bin/env python3
# ===============================================================
# Example_12_Cube_Laplacian_SurfaceTest.py
# ===============================================================

import numpy as np
from math import pi

# ---------------------------------------------------------------
# 0. Import gravity model (fallback placeholder)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: 'polygravitation' not installed. Using mock model.\n")
    class PolyhedronGravitation:
        def __init__(self, **kwargs): pass
        def potential(self, point): return 0.0
        def acceleration(self, point): return np.zeros(3)
        def gravity_tensor(self, point): return np.zeros((3, 3))


# ===============================================================
# 1. Cube Geometry
# ===============================================================
def create_cube_vertices_faces():
    V = np.array([
        [0.,0.,0.], [2.,0.,0.], [2.,2.,0.], [0.,2.,0.],
        [0.,0.,2.], [2.,0.,2.], [2.,2.,2.], [0.,2.,2.]
    ])

    F = np.array([
        [0,1,2],[0,2,3],
        [4,6,5],[4,7,6],
        [0,5,1],[0,4,5],
        [3,2,6],[3,6,7],
        [0,3,7],[0,7,4],
        [1,5,6],[1,6,2],
    ])

    return V, F


# ===============================================================
# 2. Symbolic coordinate formatter
# ===============================================================
def fmt_symbolic(name, p, eps):
    """Return symbolic coordinate representation."""
    # symbolic expressions for epsilon-shifted points
    if "exterior" in name:
        return f"({p[0]-eps:+g}+ε, {p[1]-eps:+g}+ε, {p[2]-eps:+g}+ε)"
    if "interior" in name:
        # sign depends on component
        sx = "+ε" if p[0] > np.round(p[0]) else "−ε"
        sy = "+ε" if p[1] > np.round(p[1]) else "−ε"
        sz = "+ε" if p[2] > np.round(p[2]) else "−ε"
        return f"({int(np.round(p[0]))}{sx}, {int(np.round(p[1]))}{sy}, {int(np.round(p[2]))}{sz})"

    # original points (integers)
    return f"({int(p[0])}, {int(p[1])}, {int(p[2])})"


# ===============================================================
# 3. Main Execution
# ===============================================================
if __name__ == "__main__":
    G = 1.0
    rho = 1.0
    eps = 1e-10

    V, F = create_cube_vertices_faces()

    model = PolyhedronGravitation(vertices=V, faces=F,
                                  G=G, density=rho, orient_faces=True)

    # -----------------------------------------------------------
    # Test points
    # -----------------------------------------------------------
    test_points = {
        "V1 original": np.array([2., 0., 2.]),
        "V1 exterior": np.array([2.+eps, 0.+eps, 2.+eps]),
        "V1 interior": np.array([2.-eps, 0.+eps, 2.-eps]),

        "V2 original": np.array([2., 2., 2.]),
        "V2 exterior": np.array([2.+eps, 2.+eps, 2.+eps]),
        "V2 interior": np.array([2.-eps, 2.-eps, 2.-eps]),

        "F1 original": np.array([1., 1., 2.]),
        "F1 interior": np.array([1., 1., 2.-eps]),
        "F1 exterior": np.array([1., 1., 2.+eps]),

        "F2 original": np.array([2., 1., 1.]),
        "F2 interior": np.array([2.-eps, 1., 1.]),
        "F2 exterior": np.array([2.+eps, 1., 1.]),
    }

    # -----------------------------------------------------------
    # Table Header
    # -----------------------------------------------------------
    header = f"{'Point ID':<18s} | {'Test Point (symbolic)':<35s} | {'Laplacian L':>12s}"
    sep = "-" * len(header)

    print(sep)
    print(header)
    print(sep)

    lines = [sep, header, sep]

    # -----------------------------------------------------------
    # Compute Laplacian
    # -----------------------------------------------------------
    for name, p in test_points.items():
        try:
            Γ = model.gravity_tensor(p)
            L = np.trace(Γ) / (4 * pi * G * rho)

            symbolic = fmt_symbolic(name, p, eps)

            line = f"{name:<18s} | {symbolic:<35s} | {L:+.4e}"
            print(line)
            lines.append(line)

        except Exception as e:
            symbolic = fmt_symbolic(name, p, eps)
            line = f"{name:<18s} | {symbolic:<35s} | FAILED: {e}"
            print(line)
            lines.append(line)

    print(sep)
    lines.append(sep)
    lines.append("Computation complete.")

    # -----------------------------------------------------------
    # Save file
    # -----------------------------------------------------------
    fname = "Example_12_Cube_Laplacian_SurfaceTest.txt"
    with open(fname, "w") as f:
        for Lline in lines:
            f.write(Lline + "\n")

    print(f"\nSaved results to '{fname}'.")


-----------------------------------------------------------------------
Point ID           | Test Point (symbolic)               |  Laplacian L
-----------------------------------------------------------------------
V1 original        | (2, 0, 2)                           | -1.2500e-01
V1 exterior        | (+2+ε, +0+ε, +2+ε)                  | +0.0000e+00
V1 interior        | (2−ε, 0+ε, 2−ε)                     | -1.0000e+00
V2 original        | (2, 2, 2)                           | -1.2500e-01
V2 exterior        | (+2+ε, +2+ε, +2+ε)                  | +8.8349e-18
V2 interior        | (2−ε, 2−ε, 2−ε)                     | -1.0000e+00
F1 original        | (1, 1, 2)                           | -5.0000e-01
F1 interior        | (1−ε, 1−ε, 2−ε)                     | -1.0000e+00
F1 exterior        | (+1+ε, +1+ε, +2+ε)                  | -3.5335e-07
F2 original        | (2, 1, 1)                           | -5.0000e-01
F2 interior        | (2−ε, 1−ε, 1−ε)                     | -1.0000e+00
F2 

In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_13_Icosahedron_Laplacian_TheoryVsNumerical.py
# ===============================================================

import numpy as np
from math import pi, acos, sqrt
from polygravitation import PolyhedronGravitation

# ===============================================================
# 1. Build regular icosahedron
# ===============================================================
def create_icosahedron():
    φ = (1 + sqrt(5)) / 2
    V = np.array([
        [-1,  φ, 0], [ 1,  φ, 0], [-1, -φ, 0], [ 1, -φ, 0],
        [0, -1,  φ], [0,  1,  φ], [0, -1, -φ], [0,  1, -φ],
        [ φ, 0, -1], [ φ, 0,  1], [-φ, 0, -1], [-φ, 0,  1]
    ], float)

    F = np.array([
        [0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11],
        [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6], [7, 1, 8],
        [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9],
        [4, 9, 5], [2, 4, 11], [6, 2, 10], [8, 6, 7], [9, 8, 1]
    ], int)

    return V, F


# ===============================================================
# 2. Theoretical Laplacian values
# ===============================================================
def theoretical_values():
    # Dihedral angle alpha
    α = acos(-sqrt(5)/3)

    # Solid angle at vertex of regular icosahedron
    ω = 5 * acos((sqrt(5)+1)/4) - pi

    return {
        "face": 0.5,
        "edge": α/(2*pi),
        "vertex": ω/(4*pi),
        "α": α,
        "ω": ω
    }


# ===============================================================
# 3. Formatting helper
# ===============================================================
def fmt_symbolic(text):
    return text


def laplace_num(model, p):
    Γ = model.gravity_tensor(p)
    return np.trace(Γ)/(4*pi*1*1)


# ===============================================================
# 4. Main
# ===============================================================
if __name__ == "__main__":

    eps = 1e-10

    V, F = create_icosahedron()
    model = PolyhedronGravitation(vertices=V, faces=F, G=1, density=1, orient_faces=True)

    theory = theoretical_values()

    print("\nTHEORETICAL LAPLACIANS")
    print("----------------------")
    print(f"Face:   0.5")
    print(f"Edge:   {theory['edge']:.6f}")
    print(f"Vertex: {theory['vertex']:.6f}")
    print(f"α (dihedral angle): {theory['α']:.6f} rad")
    print(f"ω (solid angle):    {theory['ω']:.6f} sr\n")

    # -----------------------------------------------------------
    # Test points
    # -----------------------------------------------------------
    v0 = V[0]
    v11 = V[11]
    v5 = V[5]

    face_center = (v0 + v11 + v5)/3
    edge_mid = (v0 + v11)/2
    vertex = v0

    test_points = {
        "Face original": face_center,
        "Face exterior": face_center + eps * np.array([1,1,1]),
        "Face interior": face_center - eps * np.array([1,1,1]),

        "Edge original": edge_mid,
        "Edge exterior": edge_mid + eps * np.array([1,1,1]),
        "Edge interior": edge_mid - eps * np.array([1,1,1]),

        "Vertex original": vertex,
        "Vertex exterior": vertex + eps * np.array([1,1,1]),
        "Vertex interior": vertex - eps * np.array([1,1,1]),
    }

    print("NUMERICAL VS THEORETICAL LAPLACE VALUES")
    print("---------------------------------------")
    print(f"{'Point':<20} | {'L_num':>12} | {'L_theory':>12}")
    print("-"*50)

    for name, p in test_points.items():
        L_num = laplace_num(model, p)

        if "Face" in name:
            L_th = theory["face"]
        elif "Edge" in name:
            L_th = theory["edge"]
        elif "Vertex" in name:
            L_th = theory["vertex"]

        print(f"{name:<20} | {L_num:+.6f} | {L_th:+.6f}")



In [ ]:
#!/usr/bin/env python3
# ===============================================================
# Example_13_Icosahedron_Laplacian_SurfaceTest.py
# ===============================================================
# Homogeneous Icosahedron – Theoretical vs Numerical Laplacian
#
# For a homogeneous polyhedron:
#   L_face   = 1/2            on a face
#   L_edge   = α / (2π)       on an edge
#   L_vertex = ω / (4π)       on a vertex
#
# where:
#   α  = dihedral angle (radians) along the edge
#   ω  = solid angle (steradians) at the vertex
#
# This script:
#   • Builds a transformed icosahedron with one face exactly on
#       (0,0,0), (1,0,0), (0,1,0)  (z = 0 plane, integer coords)
#   • Chooses three exact points on that face:
#       - Vertex: (0, 0, 0)
#       - Edge:   (1/2, 0, 0)
#       - Face:   (1/4, 1/4, 0)  (interior point, not the center)
#   • Computes theoretical L for face/edge/vertex
#   • Computes numerical L = trace(Γ) / (4π G ρ)
#   • Prints a table and saves it to a .txt file
# ===============================================================

import numpy as np
from math import pi, sqrt, acos

# ---------------------------------------------------------------
# 0. Import gravity model (fallback placeholder)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: 'polygravitation' not installed. Using mock model.\n")

    class PolyhedronGravitation:
        def __init__(self, **kwargs):
            pass
        def potential(self, point): return 0.0
        def acceleration(self, point): return np.zeros(3)
        def gravity_tensor(self, point): return np.zeros((3, 3))


# ===============================================================
# 1. Build standard & transformed icosahedron
# ===============================================================
def create_standard_icosahedron():
    """Standard icosahedron (unscaled) with faces as in many references."""
    φ = (1.0 + sqrt(5.0)) / 2.0
    V = np.array([
        [-1,  φ, 0], [ 1,  φ, 0], [-1, -φ, 0], [ 1, -φ, 0],
        [0, -1,  φ], [0,  1,  φ], [0, -1, -φ], [0,  1, -φ],
        [ φ, 0, -1], [ φ, 0,  1], [-φ, 0, -1], [-φ, 0,  1]
    ], dtype=float)

    F = np.array([
        [0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11],
        [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6], [7, 1, 8],
        [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9],
        [4, 9, 5], [2, 4, 11], [6, 2, 10], [8, 6, 7], [9, 8, 1]
    ], dtype=np.int32)

    return V, F


def normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def create_transformed_icosahedron():
    """
    Transform the standard icosahedron so that the face (0,11,5)
    becomes exactly:
        (0,0,0), (1,0,0), (0,1,0)
    which lies on the z=0 plane with integer coordinates.
    """
    V_std, F = create_standard_icosahedron()

    # Source face vertices
    s1, s2, s3 = V_std[[0, 11, 5]]

    # Target face vertices (exact integers, parallel to xy-plane)
    t1 = np.array([0.0, 0.0, 0.0])
    t2 = np.array([1.0, 0.0, 0.0])
    t3 = np.array([0.0, 1.0, 0.0])

    # Build local frames for source & target
    u_s = normalize(s2 - s1)
    w_s = normalize(np.cross(u_s, s3 - s1))
    v_s = np.cross(w_s, u_s)
    A = np.c_[u_s, v_s, w_s]

    u_t = normalize(t2 - t1)  # (1,0,0)
    w_t = normalize(np.cross(u_t, t3 - t1))  # (0,0,1)
    v_t = np.cross(w_t, u_t)  # (0,1,0)
    B = np.c_[u_t, v_t, w_t]

    # Uniform scaling: match edge length s1-s2 to t1-t2 (length 1)
    scale = np.linalg.norm(t2 - t1) / np.linalg.norm(s2 - s1)

    # Rotation matrix
    R = B @ A.T

    # Apply similarity transform
    V_trans = np.array([(R @ (p - s1)) * scale + t1 for p in V_std])

    # Force the chosen face vertices to be *exactly* integer coordinates
    V_trans[0] = t1
    V_trans[11] = t2
    V_trans[5] = t3

    return V_trans, F


# ===============================================================
# 2. Geometry helpers: dihedral and solid angle
# ===============================================================
def dihedral_angle_for_edge(V, F, i, j):
    """Compute internal dihedral angle α (in radians) along edge (i,j)."""
    # Find the two faces that share edge (i,j)
    faces_with_edge = []
    for f in F:
        if i in f and j in f:
            faces_with_edge.append(f)
    if len(faces_with_edge) != 2:
        raise RuntimeError(f"Edge ({i},{j}) does not have exactly two adjacent faces.")

    def face_normal(face):
        a, b, c = V[face]
        n = np.cross(b - a, c - a)
        return normalize(n)

    n1 = face_normal(faces_with_edge[0])
    n2 = face_normal(faces_with_edge[1])

    cos_theta = np.clip(abs(np.dot(n1, n2)), -1.0, 1.0)
    theta = acos(cos_theta)  # angle between normals
    alpha = pi - theta       # internal dihedral angle
    return alpha


def solid_angle_at_vertex(V, F, vidx):
    """
    Compute solid angle ω (steradians) at vertex 'vidx' by building the
    spherical polygon from its neighboring vertices and triangulating.
    """
    # Faces incident to vertex
    incident_faces = [face for face in F if vidx in face]

    # Neighbor vertices around vidx
    neighbors = set()
    for face in incident_faces:
        for v in face:
            if v != vidx:
                neighbors.add(v)
    neighbors = list(neighbors)

    # Build adjacency between neighbors using faces (each face gives an edge)
    adj = {n: set() for n in neighbors}
    for face in incident_faces:
        others = [v for v in face if v != vidx]
        if len(others) == 2:
            a, b = others
            adj[a].add(b)
            adj[b].add(a)

    # Order neighbors around the vertex to form a ring
    start = neighbors[0]
    ring = [start]
    prev = None
    curr = start
    while True:
        candidates = adj[curr] - ({prev} if prev is not None else set())
        if not candidates:
            break
        nxt = candidates.pop()
        if nxt == start:
            break
        ring.append(nxt)
        prev, curr = curr, nxt

        if len(ring) > len(neighbors):
            break  # safety

    # Triangulate the polygon using fan from ring[0]
    v0 = V[vidx]
    ω = 0.0

    def solid_angle_tri(a, b, c):
        # solid angle of cone defined by vectors a,b,c from the origin
        la, lb, lc = np.linalg.norm(a), np.linalg.norm(b), np.linalg.norm(c)
        numerator = np.dot(a, np.cross(b, c))
        denom = (la * lb * lc +
                 np.dot(a, b) * lc +
                 np.dot(b, c) * la +
                 np.dot(c, a) * lb)
        return 2.0 * np.arctan2(abs(numerator), denom)

    for k in range(1, len(ring) - 1):
        r1 = V[ring[0]] - v0
        r2 = V[ring[k]] - v0
        r3 = V[ring[k + 1]] - v0
        ω += solid_angle_tri(r1, r2, r3)

    return ω


# ===============================================================
# 3. Laplacian helper
# ===============================================================
def numerical_laplacian(model, p):
    Γ = model.gravity_tensor(p)
    return np.trace(Γ) / (4.0 * pi * 1.0 * 1.0)  # G = 1, rho = 1


# ===============================================================
# 4. Main execution
# ===============================================================
if __name__ == "__main__":
    G = 1.0
    rho = 1.0

    V, F = create_transformed_icosahedron()
    model = PolyhedronGravitation(vertices=V, faces=F, G=G, density=rho, orient_faces=True)

    # -----------------------------------------------------------
    # Choose the special face (mapped to z=0 with integers)
    # -----------------------------------------------------------
    face_idx = 0          # this is [0, 11, 5] in F
    v0_idx, v11_idx, v5_idx = F[face_idx]

    # Sanity: those should be 0, 11, 5
    # (we rely on create_transformed_icosahedron for that)
    # Vertex positions (after transform)
    v0 = V[v0_idx]   # expected (0,0,0)
    v11 = V[v11_idx] # expected (1,0,0)
    v5 = V[v5_idx]   # expected (0,1,0)

    # -----------------------------------------------------------
    # Theoretical values
    # -----------------------------------------------------------
    # Edge: use edge (0, 11)
    alpha = dihedral_angle_for_edge(V, F, v0_idx, v11_idx)
    # Vertex: use vertex 0
    omega = solid_angle_at_vertex(V, F, v0_idx)

    L_face_theory = 0.5
    L_edge_theory = alpha / (2.0 * pi)
    L_vertex_theory = omega / (4.0 * pi)

    # -----------------------------------------------------------
    # Exact test points on the mapped face (z = 0)
    # -----------------------------------------------------------
    tests = [
        ("Vertex V0",   "vertex", np.array([0.0, 0.0, 0.0]),     "(0, 0, 0)",          L_vertex_theory),
        ("Edge V0-V11", "edge",   np.array([0.5, 0.0, 0.0]),     "(1/2, 0, 0)",        L_edge_theory),
        ("Face point",  "face",   np.array([0.25, 0.25, 0.0]),   "(1/4, 1/4, 0)",      L_face_theory),
    ]

    # -----------------------------------------------------------
    # Print & save table
    # -----------------------------------------------------------
    header = (f"{'Point ID':<14s} | {'Type':<8s} | "
              f"{'Test Point (symbolic)':<20s} | {'L_num':>10s} | {'L_theory':>10s}")
    sep = "-" * len(header)

    print(sep)
    print(header)
    print(sep)

    lines = [sep, header, sep]

    for name, ptype, p, symbolic, L_th in tests:
        L_num = numerical_laplacian(model, p)
        line = (f"{name:<14s} | {ptype:<8s} | {symbolic:<20s} | "
                f"{L_num:+.4e} | {L_th:+.4e}")
        print(line)
        lines.append(line)

    print(sep)
    lines.append(sep)
    lines.append("Computation complete.")

    # Save to .txt file
    fname = "Example_13_Icosahedron_Laplacian_SurfaceTest.txt"
    with open(fname, "w") as f:
        for L in lines:
            f.write(L + "\n")

    print(f"\nResults saved to '{fname}'.")


In [ ]:
 #!/usr/bin/env python3
# ===============================================================
# Example_13_Icosahedron_Laplacian_SurfaceTest.py
# ===============================================================
# Uses a transformed icosahedron whose selected face is mapped to:
#   (0, 0, 0)
#   (2, 0, 0)
#   (0, 1, 0)
# Test points (exact integer):
#   Vertex: (0,0,0)
#   Edge:   (1,0,0)
#   Face:   (1,1,0)
# ===============================================================

import numpy as np
from math import pi, sqrt, acos

# ---------------------------------------------------------------
# 0. Import gravity model (fallback)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: polygravitation not installed. Using mock model.")
    class PolyhedronGravitation:
        def __init__(self, **kwargs): pass
        def gravity_tensor(self, p): return np.zeros((3,3))


# ===============================================================
# 1. Standard icosahedron
# ===============================================================
def create_standard_icosahedron():
    φ = (1 + sqrt(5)) / 2
    V = np.array([
        [-1,  φ, 0], [ 1,  φ, 0], [-1, -φ, 0], [ 1, -φ, 0],
        [0, -1,  φ], [0,  1,  φ], [0, -1, -φ], [0,  1, -φ],
        [ φ, 0, -1], [ φ, 0,  1], [-φ, 0, -1], [-φ, 0,  1]
    ], float)

    F = np.array([
        [0,11,5],[0,5,1],[0,1,7],[0,7,10],[0,10,11],
        [1,5,9],[5,11,4],[11,10,2],[10,7,6],[7,1,8],
        [3,9,4],[3,4,2],[3,2,6],[3,6,8],[3,8,9],
        [4,9,5],[2,4,11],[6,2,10],[8,6,7],[9,8,1]
    ], int)
    return V, F


def normalize(v):
    n = np.linalg.norm(v)
    return v/n if n>0 else v


# ===============================================================
# 2. Map face (0,11,5) → (0,0,0), (2,0,0), (0,1,0)
# ===============================================================
def create_transformed_icosahedron():
    V_std, F = create_standard_icosahedron()

    # Source triangle
    s1, s2, s3 = V_std[[0,11,5]]

    # Target triangle (integers)
    t1 = np.array([0.,0.,0.])
    t2 = np.array([2.,0.,0.])
    t3 = np.array([0.,2.,0.])

    # Build source frame
    u_s = normalize(s2 - s1)
    w_s = normalize(np.cross(u_s, s3 - s1))
    v_s = np.cross(w_s, u_s)
    A = np.c_[u_s, v_s, w_s]

    # Target frame
    u_t = normalize(t2 - t1)
    w_t = normalize(np.cross(u_t, t3 - t1))
    v_t = np.cross(w_t, u_t)
    B = np.c_[u_t, v_t, w_t]

    # Scale edge length s1-s2 → length 2.0
    scale = 2.0 / np.linalg.norm(s2 - s1)
    R = B @ A.T

    # Transform all vertices
    V_new = np.array([(R @ (p - s1)) * scale + t1 for p in V_std])

    # Force exact integers on the mapped face
    V_new[0]  = t1
    V_new[11] = t2
    V_new[5]  = t3

    return V_new, F


# ===============================================================
# 3. Geometry helpers
# ===============================================================
def dihedral_angle_for_edge(V, F, i, j):
    faces = [f for f in F if i in f and j in f]
    def n(f):
        a,b,c = V[f]
        return normalize(np.cross(b-a, c-a))
    n1,n2 = n(faces[0]), n(faces[1])
    θ = acos(np.clip(abs(np.dot(n1,n2)), -1,1))
    return pi - θ  # internal dihedral angle


def solid_angle_at_vertex(V, F, vidx):
    # Collect incident neighbors
    neigh = []
    for f in F:
        if vidx in f:
            for v in f:
                if v != vidx and v not in neigh:
                    neigh.append(v)

    # Build ring ordering (simple cyclic)
    ring = []
    used = set()
    curr = neigh[0]
    ring.append(curr)
    used.add(curr)

    for _ in range(len(neigh)-1):
        for v in neigh:
            if v not in used:
                ring.append(v)
                used.add(v)
                break

    # Fan triangulation
    v0 = V[vidx]
    total = 0.0
    def Ω(a,b,c):
        la,lb,lc = np.linalg.norm(a),np.linalg.norm(b),np.linalg.norm(c)
        num = np.dot(a, np.cross(b,c))
        den = la*lb*lc + np.dot(a,b)*lc + np.dot(b,c)*la + np.dot(c,a)*lb
        return 2*np.arctan2(abs(num), den)

    for k in range(1, len(ring)-1):
        r1 = V[ring[0]] - v0
        r2 = V[ring[k]] - v0
        r3 = V[ring[k+1]] - v0
        total += Ω(r1,r2,r3)

    return total


# ===============================================================
# 4. Numerical Laplacian
# ===============================================================
def L_num(model, p):
    Γ = model.gravity_tensor(p)
    return np.trace(Γ) / (4*pi)


# ===============================================================
# 5. Main
# ===============================================================
if __name__ == "__main__":
    eps = 1e-10

    V, F = create_transformed_icosahedron()
    model = PolyhedronGravitation(vertices=V, faces=F, G=1, density=1, orient_faces=True)

    # The mapped face is (0, 11, 5)
    vid0, vid11, vid5 = F[0]

    # Compute theoretical terms
    α = dihedral_angle_for_edge(V, F, vid0, vid11)
    ω = solid_angle_at_vertex(V, F, vid0)

    L_face_th   = 0.5
    L_edge_th   = α / (2*pi)
    L_vertex_th = ω / (4*pi)

    # Test points
    tests = [
        ("Vertex", (0,0,0), L_vertex_th),
        ("Edge",   (1,0,0), L_edge_th),
        ("Face",   (1,1,0), L_face_th),
    ]

    # Header
    header = (f"{'ID':<10} | {'Symbolic':<25} | {'L_num':>10} | {'L_theory':>10}")
    sep = "-" * len(header)
    print(sep)
    print(header)
    print(sep)

    lines = [sep, header, sep]

    # Evaluate with ε-shifts
    for name, p0, Lth in tests:
        px,py,pz = p0

        # Exact point
        p_exact = np.array(p0, float)
        L0 = L_num(model, p_exact)

        # Exterior
        p_ext = np.array([px+eps, py+eps, pz+eps])
        Lext = L_num(model, p_ext)

        # Interior
        p_int = np.array([px-eps, py-eps, pz-eps])
        Lint = L_num(model, p_int)

        rows = [
            (f"{name} orig", f"({px},{py},{pz})",     L0,   Lth),
            (f"{name} ext ", f"({px}+ε,{py}+ε,{pz}+ε)", Lext, Lth),
            (f"{name} int ", f"({px}-ε,{py}-ε,{pz}-ε)", Lint, Lth),
        ]

        for rname, symb, Ln, Lth in rows:
            line = f"{rname:<10} | {symb:<25} | {Ln:+.4e} | {Lth:+.4e}"
            print(line)
            lines.append(line)

    print(sep)
    lines.append(sep)

    with open("Example_13_Icosahedron_Laplacian_SurfaceTest.txt", "w") as f:
        f.write("\n".join(lines))

    print("\nSaved to Example_13_Icosahedron_Laplacian_SurfaceTest.txt\n")


In [1]:
#!/usr/bin/env python3
# ===============================================================
# Example_Tetrahedron_Laplacian_SurfaceTest.py
# ===============================================================
# Laplacian surface-limit test for a tetrahedron:
#
# Vertices:
#   A = (0,0,0)
#   B = (0,2,0)
#   C = (2,0,0)
#   D = (0,0,2)
#
# Test points (on the surface):
#   Vertex: A      = (0,0,0)
#   Edge:   AC mid = (1,0,0)
#   Face:   ABC    = (1,1,0)
#
# L (theoretical):
#   On face   : L = 1/2
#   On edge   : L = alpha / (2π)
#   On vertex : L = omega / (4π)
# ===============================================================

import numpy as np
from math import pi, acos, degrees

# ---------------------------------------------------------------
# 0. Import gravity model (fallback placeholder)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print(" Warning: 'polygravitation' not installed. Using mock model.\n")
    class PolyhedronGravitation:
        def __init__(self, **kwargs): pass
        def potential(self, point): return 0.0
        def acceleration(self, point): return np.zeros(3)
        def gravity_tensor(self, point): return np.zeros((3, 3))


# ===============================================================
# 1. Tetrahedron Geometry
# ===============================================================
def create_tetra_vertices_faces():
    V = np.array([
        [0., 0., 0.],  # A = 0
        [0., 3., 0.],  # B = 1
        [3., 0., 0.],  # C = 2
        [0., 0., 2.],  # D = 3
    ], dtype=float)

    F = np.array([
        [0, 1, 2],  # ABC
        [0, 3, 1],  # ABD
        [0, 2, 3],  # ACD
        [1, 3, 2],  # BCD
    ], dtype=int)

    return V, F


def normalize(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


# ===============================================================
# 2. Angle Computations (alpha: dihedral, omega: solid angle)
# ===============================================================
def dihedral_angle(V, F, i, j):
    """Internal dihedral angle alpha (radians) along edge (i,j)."""
    faces = [f for f in F if i in f and j in f]
    if len(faces) != 2:
        raise ValueError("Edge does not have exactly two adjacent faces.")

    def face_normal(face):
        a, b, c = V[face]
        return normalize(np.cross(b - a, c - a))

    n1 = face_normal(faces[0])
    n2 = face_normal(faces[1])

    cos_theta = np.clip(np.dot(n1, n2), -1.0, 1.0)
    theta = acos(cos_theta)
    alpha = pi - theta  # internal dihedral
    return alpha


def solid_angle_at_vertex(V, vid):
    """
    Solid angle omega (steradians) at vertex 'vid' for a tetrahedron,
    using the 3 edge vectors from that vertex.
    """
    # Collect the 3 other vertices
    others = [k for k in range(V.shape[0]) if k != vid]
    if len(others) != 3:
        raise ValueError("Tetrahedron vertex must have exactly 3 opposite vertices.")

    A = V[vid]
    u = V[others[0]] - A
    v = V[others[1]] - A
    w = V[others[2]] - A

    lu = np.linalg.norm(u)
    lv = np.linalg.norm(v)
    lw = np.linalg.norm(w)

    det_uvw = np.dot(u, np.cross(v, w))

    denom = (
        lu * lv * lw +
        np.dot(u, v) * lw +
        np.dot(v, w) * lu +
        np.dot(w, u) * lv
    )

    omega = 2.0 * np.arctan2(abs(det_uvw), denom)
    return omega


def laplacian_from_tensor(Gamma):
    return np.trace(Gamma) / (4.0 * pi)


# ===============================================================
# 3. Symbolic coordinate formatter (cube style)
# ===============================================================
def fmt_symbolic(name, p, eps):
    """Return symbolic coordinate representation."""
    if "exterior" in name:
        # p is already shifted by +eps numerically; show base+ε
        return f"({p[0]-eps:+g}+ε, {p[1]-eps:+g}+ε, {p[2]-eps:+g}+ε)"
    if "interior" in name:
        # p is base-eps numerically; show base±ε form
        sx = "+ε" if p[0] > 0 else "−ε"
        sy = "+ε" if p[1] > 0 else "−ε"
        sz = "+ε" if p[2] > 0 else "−ε"
        return f"({int(round(p[0]))}{sx}, {int(round(p[1]))}{sy}, {int(round(p[2]))}{sz})"
    # original (integers)
    return f"({int(p[0])}, {int(p[1])}, {int(p[2])})"


# ===============================================================
# 4. Main Execution
# ===============================================================
if __name__ == "__main__":
    G = 1.0
    rho = 1.0
    eps = 1e-10

    V, F = create_tetra_vertices_faces()

    model = PolyhedronGravitation(vertices=V, faces=F,
                                  G=G, density=rho, orient_faces=True)

    # -----------------------------------------------------------
    # Theoretical L values (from Poisson/Laplace surface limits)
    # L_face   = 1/2
    # L_edge   = alpha / (2π)
    # L_vertex = omega / (4π)
    # -----------------------------------------------------------

    # Vertex: use A = index 0
    omega = solid_angle_at_vertex(V, vid=0)
    L_vertex_theory = omega / (4.0 * pi)

    # Edge: use AC = edge (0,2)
    alpha = dihedral_angle(V, F, i=0, j=2)
    L_edge_theory = alpha / (2.0 * pi)

    # Face (ABC): L_face = 1/2
    L_face_theory = 0.5

    # Print angles with units
    print("\n==============================================")
    print("Tetrahedron Laplacian Surface Test")
    print("==============================================\n")
    print(f"alpha (dihedral angle along AC) = {alpha:.10f} rad  = {degrees(alpha):.6f} deg")
    print(f"omega (solid angle at A)        = {omega:.10f} sr\n")

    # -----------------------------------------------------------
    # Test points (on surface + epsilon shifts)
    # -----------------------------------------------------------
    test_points = {
        # Vertex A = (0,0,0)
        "V original": np.array([0.0, 0.0, 0.0]),
        "V exterior": np.array([eps,  eps,  eps]),
        "V interior": np.array([-eps, -eps, -eps]),

        # Edge AC midpoint = (1,0,0)
        "E original": np.array([1.0, 0.0, 0.0]),
        "E interior": np.array([1.0, eps,  0.0]),   # y>0 → interior
        "E exterior": np.array([1.0, -eps, 0.0]),   # y<0 → exterior

        # Face ABC interior = (1,1,0)
        "F original": np.array([1.0, 1.0, 0.0]),
        "F interior": np.array([1.0, 1.0,  eps]),   # z>0 → interior
        "F exterior": np.array([1.0, 1.0, -eps]),   # z<0 → exterior
    }

    # Map leading letter to theoretical L
    theory_map = {
        "V": L_vertex_theory,
        "E": L_edge_theory,
        "F": L_face_theory,
    }

    # -----------------------------------------------------------
    # Table Header
    # -----------------------------------------------------------
    header = (f"{'Point ID':<18s} | "
              f"{'Test Point (symbolic)':<35s} | "
              f"{'Laplacian L_num':>16s} | "
              f"{'L_theory':>10s}")
    sep = "-" * len(header)

    print(sep)
    print(header)
    print(sep)

    lines = [sep, header, sep]

    # -----------------------------------------------------------
    # Compute Laplacian for each test point
    # -----------------------------------------------------------
    for name, p in test_points.items():
        try:
            Gamma = model.gravity_tensor(p)
            L_num = laplacian_from_tensor(Gamma)
            sym = fmt_symbolic(name, p, eps)
            key = name[0]  # 'V', 'E', or 'F'
            L_th = theory_map[key]

            line = (f"{name:<18s} | {sym:<35s} | "
                    f"{L_num:+.4e}         | {L_th:+.4e}")
            print(line)
            lines.append(line)

        except Exception as e:
            sym = fmt_symbolic(name, p, eps)
            line = f"{name:<18s} | {sym:<35s} | FAILED: {e}"
            print(line)
            lines.append(line)

    print(sep)
    lines.append(sep)
    lines.append("Computation complete.")

    # -----------------------------------------------------------
    # Save file
    # -----------------------------------------------------------
    fname = "Example_Tetrahedron_Laplacian_SurfaceTest.txt"
    with open(fname, "w") as f:
        for Lline in lines:
            f.write(Lline + "\n")

    print(f"\nSaved results to '{fname}'.")



Tetrahedron Laplacian Surface Test

alpha (dihedral angle along AC) = 1.5707963268 rad  = 90.000000 deg
omega (solid angle at A)        = 1.5707963268 sr

----------------------------------------------------------------------------------------
Point ID           | Test Point (symbolic)               |  Laplacian L_num |   L_theory
----------------------------------------------------------------------------------------
V original         | (0, 0, 0)                           | -1.2500e-01         | +1.2500e-01
V exterior         | (+0+ε, +0+ε, +0+ε)                  | -1.0000e+00         | +1.2500e-01
V interior         | (0−ε, 0−ε, 0−ε)                     | -1.7670e-17         | +1.2500e-01
E original         | (1, 0, 0)                           | -2.5000e-01         | +2.5000e-01
E interior         | (1+ε, 0+ε, 0−ε)                     | -1.0000e+00         | +2.5000e-01
E exterior         | (+1+ε, -2e-10+ε, -1e-10+ε)          | +1.9314e-11         | +2.5000e-01
F original         

In [5]:
#!/usr/bin/env python3
# ===============================================================
# Example_Tetrahedron_Laplacian_SurfaceTest.py
# ===============================================================

import numpy as np
from math import pi, acos, degrees

# ---------------------------------------------------------------
# Polyhedron model import (fallback)
# ---------------------------------------------------------------
try:
    from polygravitation import PolyhedronGravitation
except ImportError:
    print("polygravitation not installed — using mock model.")
    class PolyhedronGravitation:
        def __init__(self, **kwargs): pass
        def gravity_tensor(self, p): return np.zeros((3,3))


# ===============================================================
# Geometry helper
# ===============================================================
def normalize(v):
    n = np.linalg.norm(v)
    return v/n if n > 0 else v


# ===============================================================
# Solid angle (omega) at a vertex using triple-product formula
# ===============================================================
def solid_angle_at_vertex(V, vid):
    """Solid angle omega at vertex vid (steradians)."""
    A = V[vid]
    others = [i for i in range(4) if i != vid]
    u = V[others[0]] - A
    v = V[others[1]] - A
    w = V[others[2]] - A

    lu, lv, lw = np.linalg.norm(u), np.linalg.norm(v), np.linalg.norm(w)

    detUVW = np.abs(np.dot(u, np.cross(v, w)))
    denom = (
        lu*lv*lw +
        np.dot(u, v)*lw +
        np.dot(v, w)*lu +
        np.dot(w, u)*lv
    )

    return 2.0 * np.arctan2(detUVW, denom)


# ===============================================================
# Dihedral angle alpha along edge (i,j)
# ===============================================================
def dihedral_angle(V, F, i, j):
    faces = [f for f in F if i in f and j in f]

    def face_normal(face):
        a, b, c = V[face]
        return normalize(np.cross(b - a, c - a))

    n1 = face_normal(faces[0])
    n2 = face_normal(faces[1])

    cosang = np.clip(np.dot(n1, n2), -1, 1)
    theta = acos(cosang)
    return pi - theta   # internal dihedral


# ===============================================================
# Laplacian from gravity tensor
# ===============================================================
def laplacian_from_tensor(Gamma):
    return np.trace(Gamma) / (4*pi)


# ===============================================================
# Tetrahedron: Updated integer vertices with interior integer face points
# ===============================================================
V = np.array([
    [0.,0.,0.],   # A = 0
    [0.,3.,0.],   # B = 1
    [3.,0.,0.],   # C = 2
    [0.,0.,3.]    # D = 3
], float)

F = np.array([
    [0,1,2],   # ABC
    [0,1,3],   # ABD
    [0,2,3],   # ACD
    [1,2,3],   # BCD
], int)

eps = 1e-10

model = PolyhedronGravitation(vertices=V, faces=F,
                              G=1.0, density=1.0, orient_faces=True)


# ===============================================================
# Theoretical values
# ===============================================================
# Vertex: A (0)
omega = solid_angle_at_vertex(V, 0)
L_vertex_theory = omega / (4*pi)

# Edge: choose A–C (0–2)
alpha = dihedral_angle(V, F, 0, 2)
L_edge_theory = alpha / (2*pi)

# Face: ABC interior → L = 1/2
L_face_theory = 0.5


# ===============================================================
# Test points (integer interior locations)
# ===============================================================

# Face ABC interior point:
F_face = np.array([1., 1., 0.])     # interior

# Edge AC interior point:
F_edge = np.array([1.5, 0., 0.])    # midpoint of A(0,0,0)–C(3,0,0)

# Vertex:
F_vertex = np.array([0.,0.,0.])

test_points = {
    # Vertex test
    "V original": F_vertex,
    "V exterior": F_vertex + eps*np.array([1.,1.,1.]),
    "V interior": F_vertex - eps*np.array([1.,1.,1.]),

    # Edge test (AC)
    "E original": F_edge,
    "E exterior": F_edge + eps*np.array([0.,-1.,0.]),  # outward normal for AC is -y
    "E interior": F_edge + eps*np.array([0.,+1.,0.]),  # inward

    # Face test ABC
    "F original": F_face,
    "F exterior": F_face + eps*np.array([0.,0.,-1.]),  # outward normal is -z
    "F interior": F_face + eps*np.array([0.,0.,+1.]),
}

theory_map = {
    "V": L_vertex_theory,
    "E": L_edge_theory,
    "F": L_face_theory
}


# ===============================================================
# Symbolic printing (cube style)
# ===============================================================
def fmt_symbolic(name, p, eps):
    if "original" in name:
        return f"({int(round(p[0]))}, {int(round(p[1]))}, {int(round(p[2]))})"
    if "exterior" in name:
        return f"({p[0]-eps:+g}+ε, {p[1]-eps:+g}+ε, {p[2]-eps:+g}+ε)"
    if "interior" in name:
        sx = "+ε" if "F interior" in name else ("+ε" if p[0]>0 else "−ε")
        sy = "+ε" if p[1]>0 else "−ε"
        sz = "+ε" if p[2]>0 else "−ε"
        return f"({int(round(p[0]))}{sx}, {int(round(p[1]))}{sy}, {int(round(p[2]))}{sz})"
    return "(?)"


# ===============================================================
# Print results
# ===============================================================
print("\n==============================================")
print("  Tetrahedron Laplacian Surface Test")
print("==============================================\n")
print(f"alpha (dihedral angle) = {alpha:.10f} rad  = {degrees(alpha):.6f} deg")
print(f"omega (solid angle)    = {omega:.10f} sr\n")

header = f"{'Point ID':<18s} | {'Symbolic Point':<25s} | {'L_num':>12s} | {'L_theory':>12s}"
sep = "-" * len(header)

print(sep)
print(header)
print(sep)

lines = [sep, header, sep]

for name, p in test_points.items():
    Gamma = model.gravity_tensor(p)
    Lnum = laplacian_from_tensor(Gamma)

    symb = fmt_symbolic(name, p, eps)
    Ltheory = theory_map[name[0]]

    line = f"{name:<18s} | {symb:<25s} | {Lnum:+.4e} | {Ltheory:+.4e}"
    print(line)
    lines.append(line)

print(sep)
lines.append(sep)


# ===============================================================
# Save results
# ===============================================================
fname = "Example_Tetrahedron_Laplacian_SurfaceTest.txt"
with open(fname, "w") as f:
    f.write("\n".join(lines))

print(f"\nSaved to '{fname}'.\n")



  Tetrahedron Laplacian Surface Test

alpha (dihedral angle) = 1.5707963268 rad  = 90.000000 deg
omega (solid angle)    = 1.5707963268 sr

----------------------------------------------------------------------------
Point ID           | Symbolic Point            |        L_num |     L_theory
----------------------------------------------------------------------------
V original         | (0, 0, 0)                 | -1.2500e-01 | +1.2500e-01
V exterior         | (+0+ε, +0+ε, +0+ε)        | -1.0000e+00 | +1.2500e-01
V interior         | (0−ε, 0−ε, 0−ε)           | +2.5873e-17 | +1.2500e-01
E original         | (2, 0, 0)                 | -2.5000e-01 | +2.5000e-01
E exterior         | (+1.5+ε, -2e-10+ε, -1e-10+ε) | +1.7168e-11 | +2.5000e-01
E interior         | (2+ε, 0+ε, 0−ε)           | -1.0000e+00 | +2.5000e-01
F original         | (1, 1, 0)                 | -1.0000e+00 | +5.0000e-01
F exterior         | (+1+ε, +1+ε, -2e-10+ε)    | +1.4136e-16 | +5.0000e-01
F interior         | (1+ε,